
# FINAL-FREEZE-Erweiterung: FPR, AP-Korrektur, Systemvergleich, Kaskade und Fehleranalyse

Dieses Notebook **setzt den abgeschlossenen FINAL FREEZE fort**. Es ersetzt ihn nicht.

## Ziele

1. **False-Positive-Rate vollständig ausweisen**
   - Ziel-FPR: **0,5 %, 1,0 %, 2,0 %**
   - für jede schwellenwertabhängige Ergebniszeile:
     - `target_fpr`
     - `calibration_empirical_fpr`
     - `empirical_fpr`
     - FP/TN/TP/FN

2. **AP-Vergleich IID ↔ Distribution Shift korrigieren**
   - das bisherige `IID_ORIGINAL` bleibt für operative Kennzahlen erhalten;
   - für AP wird zusätzlich ein deterministisches **50/50-IID-Set** gebildet;
   - alle bestehenden Stresssets sind ebenfalls 50/50;
   - **AP-Degradation wird ausschließlich gegen `IID_BALANCED` berechnet**;
   - zusätzlich wird als Sensitivitätsanalyse für alle Szenarien ein **gleich großes AP-Set**
     mit derselben Klassenverteilung erzeugt (`AP_EQUAL_N`).

   Wichtig: Die zentrale Korrektur ist die **gleiche Klassenprävalenz**. Eine identische
   Gesamtgröße ist für AP nicht erforderlich; die Equal-N-Auswertung dient nur als zusätzliche
   Robustheitsprüfung.

3. **FINAL FREEZE mit den vorhandenen Embeddings neu auswerten**
   - BASE / DAPT / CONTRASTIVE
   - Logistic Regression / XGBoost / MLP
   - 10 % / 25 % / 100 % Labels
   - fünf Seeds
   - IID + Temporal + Domain-OOD + Template-OOD + Domain+Template-OOD

4. **Systemebene bei 25 % Labels ergänzen**
   - `B0_STRUCT_XGB`: XGBoost auf expliziten URL-/HTML-Strukturmerkmalen
   - `T0_E2E`: RoBERTa, direkt supervised fine-getuned
   - `DAPT_E2E`: gleicher Fine-Tuning-Ablauf, Initialisierung aus DAPT-40k
   - `BASE_EMB_MLP`: Repräsentationskontrolle
   - `DAPT_EMB_MLP`: stärkere DAPT-Embedding-Systemvariante
   - `CONTRASTIVE_EMB_MLP`: negative/komplementäre Referenz

5. **Kaskade**
   - Stage 1: `B0_STRUCT_XGB` blockiert am kalibrierten FPR-Arbeitspunkt;
   - Stage 2: Ranking der **nicht blockierten** Seiten;
   - Reviewbudgets 5 %, 10 %, 20 %;
   - Stage 2 ist Review/Triage und **keine automatische Blockentscheidung**.

6. **Fehleranalyse**
   - B0 vs. T0 / DAPT-E2E / DAPT-Embedding / Kontrollen
   - Rescue, Regression, FN-/FP-Überlappung
   - jeweils inklusive der tatsächlich beobachteten FPR.

## Leakage-Regel

Kalibrierung, Modelltraining und Modellselektion verwenden **keine** OOD-/Holdout-Ergebnisse.
Die neuen Stresssets bleiben reine Evaluation. DAPT wird **nicht erneut** trainiert.


> **v2-Hotfix:** Der historische B0-Feature-Loader akzeptiert nun sowohl Dictionaries als auch feste Vektoren/Arrays und serialisierte Varianten. Bei 36-dimensionalen Vektoren werden die im früheren B0-Experiment dokumentierten 36 URL-/HTML-Merkmalsnamen verwendet. Die Werte selbst werden unverändert aus dem Split-Cache übernommen.

> **v3-Resume:** Optionaler vorheriger Hybrid-Output mit `freeze_scores/` wird automatisch aus `/kaggle/input` übernommen, sodass die bereits berechnete AP/FPR-Korrektur nicht erneut gerechnet werden muss.


> **FINAL-BACHELOR-RUN:** Dieses Notebook enthält zusätzlich den abschließenden All-Budget-Systemvergleich, seed-basierte Statistik, Kaskade, Fehleranalyse, exploratives Active Learning und automatische Thesis-Abbildungen.


In [ ]:

# ============================================================
# 00 – Imports und Konfiguration
# ============================================================
import os, gc, json, math, pickle, random, shutil, zipfile, time, warnings, hashlib
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/phreshphish_hybrid_freeze_extension")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Optionaler Resume aus einem vorherigen, abgebrochenen Hybrid-Lauf.
# Erkannt werden nur Outputs dieser Erweiterung (Signal: freeze_scores/).
def hydrate_previous_hybrid_output():
    candidates = []
    for p in INPUT_ROOT.rglob("*"):
        if p.is_dir() and (p / "freeze_scores").exists():
            candidates.append(p)

    if not candidates:
        print({"hybrid_resume": "NONE"})
        return None

    source = sorted(candidates, key=lambda p: len(str(p)))[0]
    copied = []
    for item in source.iterdir():
        target = OUTPUT_ROOT / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        elif not target.exists():
            shutil.copy2(item, target)
        copied.append(item.name)

    print({
        "hybrid_resume": "DIRECTORY",
        "source": str(source),
        "copied_top_level": copied,
    })
    return str(source)

HYBRID_RESUME_SOURCE = hydrate_previous_hybrid_output()

SEEDS = [42, 52, 62, 72, 82]
REPRESENTATIONS = ["BASE", "DAPT", "CONTRASTIVE"]
CLASSIFIERS = ["LOGREG", "XGBOOST", "MLP"]
LABEL_BUDGETS = [0.10, 0.25, 1.00]

TARGET_FPRS = [0.005, 0.010, 0.020]
PRIMARY_TARGET_FPR = 0.005

CALIBRATION_FRACTION = 0.30
CALIBRATION_SPLIT_SEED = 20260808
AP_BALANCE_SEED = 20260809
AP_EQUAL_N_SEED_BASE = 20260820

SYSTEM_LABEL_BUDGET = 0.25
REVIEW_FRACTIONS = [0.05, 0.10, 0.20]

MAX_LENGTH = 256
E2E_EPOCHS = 5
E2E_LR = 2e-5
E2E_WEIGHT_DECAY = 0.01
E2E_WARMUP_RATIO = 0.10
E2E_BATCH_CANDIDATES = [16, 8, 4]
E2E_SCORE_BATCH_SIZE = 64

FREEZE_EXPECTED_SCENARIO_ROWS = {
    "TEMPORAL": 8000,
    "DOMAIN_OOD": 3858,
    "TEMPLATE_OOD": 6832,
    "DOMAIN_TEMPLATE_OOD": 3708,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False

print({
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "output": str(OUTPUT_ROOT),
    "target_fprs": TARGET_FPRS,
    "system_budget": SYSTEM_LABEL_BUDGET,
})


In [ ]:
# ============================================================
# 00a – Erfolgreiche Hybrid-Extension aus ZIP/Dataset hydrieren
# ============================================================
# Falls der erfolgreiche Lauf `phreshphish_hybrid_freeze_extension.zip`
# als Kaggle-Input angehängt ist, werden dessen freeze_scores/ und
# system_scores/ vor dem erneuten Lauf nach /kaggle/working kopiert.
# Dadurch werden insbesondere die vorhandenen 25-%-E2E-Scores wiederverwendet.

def _looks_like_successful_hybrid(p):
    return (
        p.is_dir()
        and (p / "HYBRID_FREEZE_EXTENSION_COMPLETE.json").exists()
        and (p / "freeze_scores").exists()
        and (p / "system_scores").exists()
    )

def _copy_contents(src_dir, dst_dir):
    copied = []
    for item in src_dir.iterdir():
        target = dst_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        elif not target.exists():
            shutil.copy2(item, target)
        copied.append(item.name)
    return copied

HYBRID_SUCCESS_ROOT = None

# Direkt entpacktes Kaggle-Dataset
hybrid_candidates = [
    p for p in INPUT_ROOT.rglob("*")
    if _looks_like_successful_hybrid(p)
]
if hybrid_candidates:
    HYBRID_SUCCESS_ROOT = sorted(
        hybrid_candidates, key=lambda p: len(str(p))
    )[0]

# ZIP-Fallback
if HYBRID_SUCCESS_ROOT is None:
    for zi, z in enumerate(INPUT_ROOT.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z, "r") as zf:
                names = zf.namelist()
                if not any(
                    "HYBRID_FREEZE_EXTENSION_COMPLETE.json" in n
                    for n in names
                ):
                    continue
                tmp = Path(f"/kaggle/working/_successful_hybrid_{zi}")
                if tmp.exists():
                    shutil.rmtree(tmp)
                tmp.mkdir(parents=True, exist_ok=True)
                zf.extractall(tmp)

                if _looks_like_successful_hybrid(tmp):
                    HYBRID_SUCCESS_ROOT = tmp
                else:
                    roots = [
                        p for p in tmp.rglob("*")
                        if _looks_like_successful_hybrid(p)
                    ]
                    if roots:
                        HYBRID_SUCCESS_ROOT = sorted(
                            roots, key=lambda p: len(str(p))
                        )[0]
                if HYBRID_SUCCESS_ROOT is not None:
                    break
        except Exception:
            continue

if HYBRID_SUCCESS_ROOT is not None:
    copied = _copy_contents(HYBRID_SUCCESS_ROOT, OUTPUT_ROOT)
    print({
        "successful_hybrid_resume": "LOADED",
        "source": str(HYBRID_SUCCESS_ROOT),
        "top_level_items": copied,
    })
else:
    print({
        "successful_hybrid_resume": "NONE",
        "note": "25%-Systemlauf wird bei Bedarf erneut berechnet."
    })

In [ ]:

# ============================================================
# 01 – Eingaben automatisch finden
# ============================================================

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def all_named(name):
    return list(INPUT_ROOT.rglob(name))

def looks_like_freeze_root(p):
    return (
        p.is_dir()
        and (p / "embeddings").exists()
        and (p / "best_classifier_params.json").exists()
        and (p / "tokens").exists()
    )

# FINAL-FREEZE-Output direkt als Kaggle Dataset?
freeze_candidates = [p for p in INPUT_ROOT.rglob("*") if looks_like_freeze_root(p)]
if looks_like_freeze_root(INPUT_ROOT):
    freeze_candidates.append(INPUT_ROOT)

# Oder als ZIP?
if not freeze_candidates:
    extract_root = Path("/kaggle/working/_final_freeze_extract")
    if extract_root.exists():
        shutil.rmtree(extract_root)
    for z in INPUT_ROOT.rglob("*.zip"):
        try:
            with zipfile.ZipFile(z, "r") as zf:
                names = zf.namelist()
                if (
                    any("embeddings/" in n for n in names)
                    and any("best_classifier_params.json" in n for n in names)
                    and any("tokens/" in n for n in names)
                ):
                    extract_root.mkdir(parents=True, exist_ok=True)
                    zf.extractall(extract_root)
                    break
        except Exception:
            continue
    if looks_like_freeze_root(extract_root):
        freeze_candidates.append(extract_root)
    freeze_candidates += [p for p in extract_root.rglob("*") if looks_like_freeze_root(p)]

if not freeze_candidates:
    raise FileNotFoundError(
        "Vollständiger FINAL-FREEZE-Output fehlt. Benötigt werden mindestens "
        "embeddings/, tokens/ und best_classifier_params.json."
    )

FREEZE_ROOT = sorted(set(freeze_candidates), key=lambda p: len(str(p)))[0]

split_paths = all_named("split_roles_and_holdout_cache_v2_ram_safe.pkl")
dapt_paths = all_named("dapt40k_bundle.pkl")
if not split_paths or not dapt_paths:
    raise FileNotFoundError("ENDGAME: Split-Cache bzw. dapt40k_bundle.pkl fehlt.")

SPLIT_PATH = split_paths[0]
DAPT_BUNDLE_PATH = dapt_paths[0]

# DAPT-40k-Encoder für alle fünf Seeds
DAPT_ENCODERS = {}
for seed in SEEDS:
    candidates = []
    for p in INPUT_ROOT.rglob(f"seed_{seed}"):
        s = str(p).lower().replace("\\", "/")
        has_model = (
            (p / "model.safetensors").exists()
            or (p / "pytorch_model.bin").exists()
        )
        if (
            p.is_dir()
            and has_model
            and (p / "config.json").exists()
            and "dapt_encoders" in s
            and "40k" in s
        ):
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError(f"DAPT-40k-Encoder für Seed {seed} nicht gefunden.")
    DAPT_ENCODERS[seed] = sorted(candidates, key=lambda p: len(str(p)))[0]

# Lokales RoBERTa-base für T0
base_candidates = []
for p in INPUT_ROOT.rglob("roberta-base"):
    if not p.is_dir():
        continue
    has_model = (p / "model.safetensors").exists() or (p / "pytorch_model.bin").exists()
    if has_model and (p / "config.json").exists():
        base_candidates.append(p)

if not base_candidates:
    raise FileNotFoundError(
        "Lokales roberta-base-Verzeichnis nicht gefunden. "
        "Bitte das bisherige Offline-RoBERTa-Dataset anhängen."
    )
BASE_MODEL_DIR = sorted(base_candidates, key=lambda p: len(str(p)))[0]

print(json.dumps({
    "freeze_root": str(FREEZE_ROOT),
    "split_cache": str(SPLIT_PATH),
    "dapt_bundle": str(DAPT_BUNDLE_PATH),
    "base_model": str(BASE_MODEL_DIR),
    "dapt_encoders": {str(k): str(v) for k, v in DAPT_ENCODERS.items()},
}, indent=2))


In [ ]:

# ============================================================
# 02 – Datenrollen, 40k-aware Stressszenarien und AP-Testsets
# ============================================================

split_payload = load_pickle(SPLIT_PATH)
dapt_payload = load_pickle(DAPT_BUNDLE_PATH)

def first_existing(obj, keys):
    if not isinstance(obj, dict):
        return None
    for k in keys:
        if k in obj:
            return obj[k]
    return None

train_df = first_existing(
    split_payload, ["train_df", "train", "supervised_train", "downstream_train"]
)
val_df = first_existing(
    split_payload, ["val_df", "validation_df", "validation", "val"]
)
holdout_df = first_existing(
    split_payload, ["final_holdout_clean", "final_holdout", "holdout_df", "holdout"]
)
pretrain_df = first_existing(
    dapt_payload, ["pretrain_large_df", "frame", "pretrain_df", "pretrain"]
)
dapt_holdout = first_existing(dapt_payload, ["final_holdout_clean"])

for name, obj in [
    ("train", train_df), ("validation", val_df),
    ("holdout", holdout_df), ("pretrain40k", pretrain_df)
]:
    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f"{name}: kein DataFrame gefunden.")

train_df = train_df.reset_index(drop=True).copy()
val_df = val_df.reset_index(drop=True).copy()
holdout_df = holdout_df.reset_index(drop=True).copy()
pretrain_df = pretrain_df.reset_index(drop=True).copy()

# 40k-aware Holdout-Flags aus dem DAPT-Bundle priorisieren.
if isinstance(dapt_holdout, pd.DataFrame):
    dapt_holdout = dapt_holdout.reset_index(drop=True)
    if dapt_holdout["sha256"].astype(str).duplicated().any():
        raise RuntimeError("DAPT-Holdout-SHA256 ist nicht eindeutig.")
    lookup = dapt_holdout.copy()
    lookup.index = lookup["sha256"].astype(str)
    hs = holdout_df["sha256"].astype(str)
    for c in [
        "near_duplicate_to_development",
        "min_simhash_distance_to_development",
        "template_seen_in_development",
    ]:
        if c in lookup.columns:
            mapped = hs.map(lookup[c])
            if mapped.isna().any():
                raise RuntimeError(f"40k-aware Holdout-Flag {c} konnte nicht vollständig gemappt werden.")
            holdout_df[c] = mapped.to_numpy()

# Exakt dieselbe 30/70-Aufteilung wie im FINAL FREEZE.
cal_idx, iid_idx = train_test_split(
    np.arange(len(val_df)),
    test_size=1.0 - CALIBRATION_FRACTION,
    random_state=CALIBRATION_SPLIT_SEED,
    stratify=val_df["label"].to_numpy(),
)
calibration_df = val_df.iloc[np.sort(cal_idx)].reset_index(drop=True)
iid_df = val_df.iloc[np.sort(iid_idx)].reset_index(drop=True)

# 40k-aware Development-Identitäten
development_domains = set()
development_templates = set()
for frame in [pretrain_df, train_df, val_df]:
    development_domains.update(frame["domain"].fillna("").astype(str).tolist())
    development_templates.update(frame["template_hash"].fillna("").astype(str).tolist())
development_domains.discard("")
development_templates.discard("")

h_domain = holdout_df["domain"].fillna("").astype(str)
h_template = holdout_df["template_hash"].fillna("").astype(str)
domain_seen = h_domain.isin(development_domains).to_numpy()
template_seen = h_template.isin(development_templates).to_numpy()

if "near_duplicate_to_development" not in holdout_df.columns:
    raise RuntimeError(
        "near_duplicate_to_development fehlt. Template-OOD wird nicht mit einer "
        "schwächeren Definition rekonstruiert."
    )
near_dup = holdout_df["near_duplicate_to_development"].fillna(False).astype(bool).to_numpy()

def balanced_available_indices(frame, mask, seed):
    sub = frame.loc[np.asarray(mask)].copy()
    n0 = int((sub["label"] == 0).sum())
    n1 = int((sub["label"] == 1).sum())
    n_each = min(n0, n1)
    if n_each <= 0:
        raise RuntimeError(f"Szenario nicht balancierbar: {n0=} {n1=}")
    p0 = sub[sub["label"].eq(0)].sample(n=n_each, random_state=seed)
    p1 = sub[sub["label"].eq(1)].sample(n=n_each, random_state=seed + 1)
    return np.sort(pd.concat([p0, p1]).index.to_numpy(dtype=np.int32))

domain_new_mask = (~domain_seen) & h_domain.ne("").to_numpy()
template_ood_mask = (~template_seen) & (~near_dup) & h_template.ne("").to_numpy()
domain_template_mask = domain_new_mask & template_ood_mask

stress_indices = {
    "TEMPORAL": np.arange(len(holdout_df), dtype=np.int32),
    "DOMAIN_OOD": balanced_available_indices(holdout_df, domain_new_mask, 108),
    "TEMPLATE_OOD": balanced_available_indices(holdout_df, template_ood_mask, 109),
    "DOMAIN_TEMPLATE_OOD": balanced_available_indices(holdout_df, domain_template_mask, 110),
}

# Harte Regression-Guards des FINAL FREEZE.
for name, expected_n in FREEZE_EXPECTED_SCENARIO_ROWS.items():
    idx = stress_indices[name]
    labels = holdout_df.iloc[idx]["label"].to_numpy(dtype=int)
    actual_n = len(idx)
    if actual_n != expected_n:
        raise RuntimeError(f"{name}: {actual_n} statt eingefroren {expected_n}.")
    if int((labels == 0).sum()) != int((labels == 1).sum()):
        raise RuntimeError(f"{name}: nicht 50/50 balanciert.")

# ------------------------------------------------------------
# AP-Korrektur
# ------------------------------------------------------------
# IID_ORIGINAL: für operative Kennzahlen unverändert beibehalten.
iid_y = iid_df["label"].to_numpy(dtype=int)
iid_pos = np.flatnonzero(iid_y == 1)
iid_neg = np.flatnonzero(iid_y == 0)
n_each_iid = min(len(iid_pos), len(iid_neg))

rng = np.random.default_rng(AP_BALANCE_SEED)
iid_neg_sample = rng.choice(iid_neg, size=n_each_iid, replace=False)
iid_pos_sample = rng.choice(iid_pos, size=n_each_iid, replace=False)
IID_BALANCED_IDX = np.sort(
    np.concatenate([iid_neg_sample, iid_pos_sample]).astype(np.int32)
)

# Primary AP comparison:
# gleiche Prävalenz (50/50), OOD-Daten vollständig behalten.
SCENARIOS = {
    "IID_ORIGINAL": ("iid", np.arange(len(iid_df), dtype=np.int32)),
    "IID_BALANCED": ("iid", IID_BALANCED_IDX),
    "TEMPORAL": ("holdout", stress_indices["TEMPORAL"]),
    "DOMAIN_OOD": ("holdout", stress_indices["DOMAIN_OOD"]),
    "TEMPLATE_OOD": ("holdout", stress_indices["TEMPLATE_OOD"]),
    "DOMAIN_TEMPLATE_OOD": ("holdout", stress_indices["DOMAIN_TEMPLATE_OOD"]),
}
AP_COMPARABLE_SCENARIOS = [
    "IID_BALANCED", "TEMPORAL", "DOMAIN_OOD",
    "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
]

# Zusätzliche Equal-N-Sensitivität:
# exakt n_each_iid Benign + n_each_iid Phishing in JEDEM AP-Szenario.
AP_EQUAL_N_INDICES = {"IID_BALANCED": IID_BALANCED_IDX}
for offset, name in enumerate([
    "TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
]):
    full_idx = stress_indices[name]
    y = holdout_df.iloc[full_idx]["label"].to_numpy(dtype=int)
    local0 = np.flatnonzero(y == 0)
    local1 = np.flatnonzero(y == 1)
    if min(len(local0), len(local1)) < n_each_iid:
        raise RuntimeError(f"{name}: zu klein für Equal-N AP mit {n_each_iid} je Klasse.")
    rr = np.random.default_rng(AP_EQUAL_N_SEED_BASE + offset)
    chosen_local = np.concatenate([
        rr.choice(local0, n_each_iid, replace=False),
        rr.choice(local1, n_each_iid, replace=False),
    ])
    AP_EQUAL_N_INDICES[name] = np.sort(full_idx[chosen_local].astype(np.int32))

# Audit
audit_rows = []
for name, (source, idx) in SCENARIOS.items():
    frame = iid_df.iloc[idx] if source == "iid" else holdout_df.iloc[idx]
    n0 = int((frame["label"] == 0).sum())
    n1 = int((frame["label"] == 1).sum())
    audit_rows.append({
        "set_family": "PRIMARY",
        "scenario": name,
        "source": source,
        "n": len(frame),
        "benign": n0,
        "phish": n1,
        "prevalence_phish": n1 / len(frame),
    })

for name, idx in AP_EQUAL_N_INDICES.items():
    if name == "IID_BALANCED":
        frame = iid_df.iloc[idx]
        source = "iid"
    else:
        frame = holdout_df.iloc[idx]
        source = "holdout"
    n0 = int((frame["label"] == 0).sum())
    n1 = int((frame["label"] == 1).sum())
    audit_rows.append({
        "set_family": "AP_EQUAL_N",
        "scenario": name,
        "source": source,
        "n": len(frame),
        "benign": n0,
        "phish": n1,
        "prevalence_phish": n1 / len(frame),
    })

testset_audit = pd.DataFrame(audit_rows)
testset_audit.to_csv(OUTPUT_ROOT / "testset_prevalence_audit.csv", index=False)
display(testset_audit)

print({
    "train": len(train_df),
    "calibration": len(calibration_df),
    "iid_original": len(iid_df),
    "iid_balanced": len(IID_BALANCED_IDX),
    "iid_balanced_each_class": n_each_iid,
    "holdout": len(holdout_df),
    "dapt40k": len(pretrain_df),
})


In [ ]:

# ============================================================
# 03 – FINAL-FREEZE Embeddings, Token-Caches und Labelbudgets
# ============================================================

with open(FREEZE_ROOT / "best_classifier_params.json", "r", encoding="utf-8") as f:
    BEST_PARAMS = json.load(f)
if "hidden_layer_sizes" in BEST_PARAMS["MLP"]:
    BEST_PARAMS["MLP"]["hidden_layer_sizes"] = tuple(
        BEST_PARAMS["MLP"]["hidden_layer_sizes"]
    )

EMBED_ROOT = FREEZE_ROOT / "embeddings"
TOKEN_ROOT = FREEZE_ROOT / "tokens"

def emb_path(rep, seed, split):
    seed_key = "shared" if rep == "BASE" else f"seed_{seed}"
    return EMBED_ROOT / rep.lower() / seed_key / f"{split}.npy"

def load_emb(rep, seed, split):
    effective_seed = SEEDS[0] if rep == "BASE" else seed
    p = emb_path(rep, effective_seed, split)
    if not p.exists():
        raise FileNotFoundError(p)
    return np.asarray(np.load(p, mmap_mode="r"), dtype=np.float32)

def load_token_split(folder):
    root = TOKEN_ROOT / folder
    ids = root / "input_ids.npy"
    mask = root / "attention_mask.npy"
    if not ids.exists() or not mask.exists():
        raise FileNotFoundError(f"Token-Cache fehlt: {root}")
    return {
        "input_ids": np.load(ids, mmap_mode="r"),
        "attention_mask": np.load(mask, mmap_mode="r"),
    }

TOKENS = {
    "train": load_token_split("train_4k"),
    "calibration": load_token_split("calibration"),
    "iid": load_token_split("iid_test"),
    "holdout": load_token_split("holdout_8k"),
}

expected_token_n = {
    "train": len(train_df),
    "calibration": len(calibration_df),
    "iid": len(iid_df),
    "holdout": len(holdout_df),
}
for name, arrays in TOKENS.items():
    if arrays["input_ids"].shape[0] != expected_token_n[name]:
        raise RuntimeError(
            f"Token-Cache {name}: {arrays['input_ids'].shape[0]} "
            f"statt {expected_token_n[name]}."
        )
    if arrays["input_ids"].shape[1] != MAX_LENGTH:
        raise RuntimeError(f"Token-Cache {name}: max_length ist nicht {MAX_LENGTH}.")

y_train = train_df["label"].to_numpy(dtype=int)

def nested_budget_indices(y, seed):
    rng = np.random.default_rng(seed)
    class_orders = {}
    for c in [0, 1]:
        idx = np.flatnonzero(y == c).copy()
        rng.shuffle(idx)
        class_orders[c] = idx
    result = {}
    for frac in LABEL_BUDGETS:
        parts = []
        for c in [0, 1]:
            n = len(class_orders[c]) if frac == 1.0 else max(
                1, int(round(len(class_orders[c]) * frac))
            )
            parts.append(class_orders[c][:n])
        result[frac] = np.sort(np.concatenate(parts)).astype(np.int32)
    assert set(result[0.10]).issubset(set(result[0.25]))
    assert set(result[0.25]).issubset(set(result[1.00]))
    return result

BUDGET_INDICES = {seed: nested_budget_indices(y_train, seed) for seed in SEEDS}

budget_rows = []
for seed, d in BUDGET_INDICES.items():
    for frac, idx in d.items():
        budget_rows.append({
            "seed": seed,
            "budget": frac,
            "n": len(idx),
            "benign": int((y_train[idx] == 0).sum()),
            "phish": int((y_train[idx] == 1).sum()),
        })
pd.DataFrame(budget_rows).to_csv(OUTPUT_ROOT / "label_budget_audit.csv", index=False)
display(pd.DataFrame(budget_rows))


In [ ]:

# ============================================================
# 04 – Gemeinsame Klassifikatoren, FPR-Schwellen und Metriken
# ============================================================

def build_classifier(kind, params, seed):
    if kind == "LOGREG":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(
                C=params["C"],
                max_iter=2500,
                solver="lbfgs",
                random_state=seed,
            )),
        ])
    if kind == "XGBOOST":
        return XGBClassifier(
            **params,
            subsample=0.9,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=2,
            random_state=seed,
        )
    if kind == "MLP":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", MLPClassifier(
                **params,
                activation="relu",
                solver="adam",
                batch_size=128,
                learning_rate_init=1e-3,
                max_iter=200,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=12,
                random_state=seed,
            )),
        ])
    raise KeyError(kind)

def threshold_for_target_fpr(y_true, score, target_fpr):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(score, dtype=float)
    neg = np.sort(s[y == 0])[::-1]
    if len(neg) == 0:
        raise ValueError("Keine Negativklasse für FPR-Kalibrierung.")
    allowed = int(math.floor(target_fpr * len(neg) + 1e-12))
    if allowed <= 0:
        return float(np.nextafter(neg[0], np.inf))
    if allowed >= len(neg):
        return float(-np.inf)
    return float(np.nextafter(neg[allowed], np.inf))

def metric_row(y, score, threshold):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(y, score)),
        "roc_auc": float(roc_auc_score(y, score)) if len(np.unique(y)) == 2 else np.nan,
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "empirical_fpr": float(fp / max(fp + tn, 1)),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "prevalence_phish": float((y == 1).mean()),
    }

def calibration_operating_points(ycal, cal_score):
    out = {}
    for target in TARGET_FPRS:
        thr = threshold_for_target_fpr(ycal, cal_score, target)
        m = metric_row(ycal, cal_score, thr)
        out[target] = {
            "threshold": thr,
            "calibration_empirical_fpr": m["empirical_fpr"],
            "calibration_recall": m["recall"],
        }
    return out

def scenario_score_view(iid_score, holdout_score, scenario):
    source, idx = SCENARIOS[scenario]
    if source == "iid":
        y = iid_df.iloc[idx]["label"].to_numpy(dtype=int)
        return y, np.asarray(iid_score)[idx]
    y = holdout_df.iloc[idx]["label"].to_numpy(dtype=int)
    return y, np.asarray(holdout_score)[idx]

def equal_n_score_view(iid_score, holdout_score, scenario):
    idx = AP_EQUAL_N_INDICES[scenario]
    if scenario == "IID_BALANCED":
        y = iid_df.iloc[idx]["label"].to_numpy(dtype=int)
        return y, np.asarray(iid_score)[idx]
    y = holdout_df.iloc[idx]["label"].to_numpy(dtype=int)
    return y, np.asarray(holdout_score)[idx]


In [ ]:

# ============================================================
# 05 – FINAL FREEZE korrigiert neu auswerten
#      3 Repräsentationen × 3 Classifier × 5 Seeds × 3 Budgets
# ============================================================

FREEZE_SCORE_ROOT = OUTPUT_ROOT / "freeze_scores"
FREEZE_SCORE_ROOT.mkdir(exist_ok=True)

def freeze_score_path(rep, kind, seed, frac):
    b = str(frac).replace(".", "p")
    return FREEZE_SCORE_ROOT / f"{rep}_{kind}_seed{seed}_budget{b}.npz"

freeze_rows = []

for seed in SEEDS:
    for frac in LABEL_BUDGETS:
        train_idx = BUDGET_INDICES[seed][frac]
        yb = y_train[train_idx]
        for rep in REPRESENTATIONS:
            Xtrain = load_emb(rep, seed, "train")[train_idx]
            Xcal = load_emb(rep, seed, "calibration")
            Xiid = load_emb(rep, seed, "iid")
            Xhold = load_emb(rep, seed, "holdout")
            ycal = calibration_df["label"].to_numpy(dtype=int)

            for kind in CLASSIFIERS:
                score_file = freeze_score_path(rep, kind, seed, frac)

                if score_file.exists():
                    dat = np.load(score_file)
                    cal_score = dat["cal"].astype(np.float64)
                    iid_score = dat["iid"].astype(np.float64)
                    hold_score = dat["holdout"].astype(np.float64)
                    fit_seconds = float(dat["fit_seconds"][0]) if "fit_seconds" in dat.files else np.nan
                else:
                    t0 = time.perf_counter()
                    model = build_classifier(kind, BEST_PARAMS[kind], seed)
                    model.fit(Xtrain, yb)
                    fit_seconds = time.perf_counter() - t0
                    cal_score = model.predict_proba(Xcal)[:, 1]
                    iid_score = model.predict_proba(Xiid)[:, 1]
                    hold_score = model.predict_proba(Xhold)[:, 1]
                    np.savez_compressed(
                        score_file,
                        cal=cal_score.astype(np.float32),
                        iid=iid_score.astype(np.float32),
                        holdout=hold_score.astype(np.float32),
                        fit_seconds=np.asarray([fit_seconds], dtype=np.float64),
                    )
                    del model

                ops = calibration_operating_points(ycal, cal_score)

                for target in TARGET_FPRS:
                    op = ops[target]
                    for scenario in SCENARIOS:
                        yev, sev = scenario_score_view(iid_score, hold_score, scenario)
                        freeze_rows.append({
                            "representation": rep,
                            "classifier": kind,
                            "seed": seed,
                            "label_budget": float(frac),
                            "n_train": int(len(train_idx)),
                            "scenario": scenario,
                            "n_test": int(len(yev)),
                            "target_fpr": float(target),
                            "threshold": float(op["threshold"]),
                            "calibration_empirical_fpr": float(op["calibration_empirical_fpr"]),
                            "calibration_recall": float(op["calibration_recall"]),
                            "fit_seconds": fit_seconds,
                            **metric_row(yev, sev, op["threshold"]),
                        })

                print({
                    "freeze_corrected": [rep, kind, seed, frac],
                    "rows": len(freeze_rows),
                })

            del Xtrain, Xcal, Xiid, Xhold
            gc.collect()

freeze_results = pd.DataFrame(freeze_rows)
freeze_results.to_csv(OUTPUT_ROOT / "freeze_corrected_results.csv", index=False)

# Zusammenfassung: FPR ist ausdrücklich Teil jeder Tabellenzeile.
freeze_summary = (
    freeze_results
    .groupby(
        ["representation", "classifier", "label_budget", "scenario", "target_fpr"],
        as_index=False
    )
    .agg(
        n_test=("n_test", "first"),
        prevalence_phish=("prevalence_phish", "first"),
        average_precision_mean=("average_precision", "mean"),
        average_precision_std=("average_precision", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        calibration_empirical_fpr_mean=("calibration_empirical_fpr", "mean"),
        empirical_fpr_mean=("empirical_fpr", "mean"),
        empirical_fpr_std=("empirical_fpr", "std"),
        fp_mean=("fp", "mean"),
        fn_mean=("fn", "mean"),
    )
)
freeze_summary.to_csv(OUTPUT_ROOT / "freeze_corrected_summary.csv", index=False)
display(freeze_summary.head(30))


In [ ]:

# ============================================================
# 06 – AP-Shift korrigiert:
#      primär gleiche Prävalenz; zusätzlich Equal-N Sensitivität
# ============================================================

ap_shift_rows = []

key_cols = ["representation", "classifier", "seed", "label_budget", "target_fpr"]

for keys, g in freeze_results.groupby(key_cols):
    iid_row = g[g["scenario"] == "IID_BALANCED"]
    if len(iid_row) != 1:
        raise RuntimeError(f"IID_BALANCED fehlt/dupliziert für {keys}")
    iid_row = iid_row.iloc[0]

    for scenario in ["TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"]:
        ood = g[g["scenario"] == scenario]
        if len(ood) != 1:
            raise RuntimeError(f"{scenario} fehlt/dupliziert für {keys}")
        ood = ood.iloc[0]
        ap_shift_rows.append({
            "representation": keys[0],
            "classifier": keys[1],
            "seed": keys[2],
            "label_budget": keys[3],
            "target_fpr": keys[4],
            "comparison": f"IID_BALANCED->{scenario}",
            "iid_n": int(iid_row["n_test"]),
            "ood_n": int(ood["n_test"]),
            "iid_prevalence": float(iid_row["prevalence_phish"]),
            "ood_prevalence": float(ood["prevalence_phish"]),
            "iid_ap": float(iid_row["average_precision"]),
            "ood_ap": float(ood["average_precision"]),
            "ap_delta_ood_minus_iid": float(
                ood["average_precision"] - iid_row["average_precision"]
            ),
            "iid_empirical_fpr": float(iid_row["empirical_fpr"]),
            "ood_empirical_fpr": float(ood["empirical_fpr"]),
            "empirical_fpr_delta": float(
                ood["empirical_fpr"] - iid_row["empirical_fpr"]
            ),
            "iid_recall": float(iid_row["recall"]),
            "ood_recall": float(ood["recall"]),
            "recall_delta": float(ood["recall"] - iid_row["recall"]),
        })

ap_shift = pd.DataFrame(ap_shift_rows)
ap_shift.to_csv(OUTPUT_ROOT / "ap_shift_corrected_prevalence_matched.csv", index=False)

# Equal-N-Sensitivität wird aus denselben gespeicherten Scores berechnet.
equal_n_rows = []
for seed in SEEDS:
    for frac in LABEL_BUDGETS:
        for rep in REPRESENTATIONS:
            for kind in CLASSIFIERS:
                dat = np.load(freeze_score_path(rep, kind, seed, frac))
                cal_score = dat["cal"]
                iid_score = dat["iid"]
                hold_score = dat["holdout"]
                ycal = calibration_df["label"].to_numpy(dtype=int)
                ops = calibration_operating_points(ycal, cal_score)

                for target in TARGET_FPRS:
                    op = ops[target]
                    for scenario in [
                        "IID_BALANCED", "TEMPORAL", "DOMAIN_OOD",
                        "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
                    ]:
                        yev, sev = equal_n_score_view(iid_score, hold_score, scenario)
                        equal_n_rows.append({
                            "representation": rep,
                            "classifier": kind,
                            "seed": seed,
                            "label_budget": float(frac),
                            "scenario": scenario,
                            "n_test": len(yev),
                            "target_fpr": target,
                            "threshold": op["threshold"],
                            "calibration_empirical_fpr": op["calibration_empirical_fpr"],
                            **metric_row(yev, sev, op["threshold"]),
                        })

equal_n_results = pd.DataFrame(equal_n_rows)
equal_n_results.to_csv(OUTPUT_ROOT / "ap_equal_n_sensitivity_results.csv", index=False)

equal_n_shift_rows = []
for keys, g in equal_n_results.groupby(key_cols):
    iid = g[g["scenario"] == "IID_BALANCED"].iloc[0]
    for scenario in ["TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"]:
        ood = g[g["scenario"] == scenario].iloc[0]
        equal_n_shift_rows.append({
            "representation": keys[0],
            "classifier": keys[1],
            "seed": keys[2],
            "label_budget": keys[3],
            "target_fpr": keys[4],
            "comparison": f"IID_BALANCED->{scenario}",
            "n_each_test": int(iid["n_test"]),
            "iid_ap": float(iid["average_precision"]),
            "ood_ap": float(ood["average_precision"]),
            "ap_delta_ood_minus_iid": float(ood["average_precision"] - iid["average_precision"]),
            "iid_empirical_fpr": float(iid["empirical_fpr"]),
            "ood_empirical_fpr": float(ood["empirical_fpr"]),
            "empirical_fpr_delta": float(ood["empirical_fpr"] - iid["empirical_fpr"]),
        })

equal_n_shift = pd.DataFrame(equal_n_shift_rows)
equal_n_shift.to_csv(OUTPUT_ROOT / "ap_shift_equal_n_sensitivity.csv", index=False)

print({
    "primary_ap_shift_rows": len(ap_shift),
    "equal_n_rows": len(equal_n_results),
    "equal_n_shift_rows": len(equal_n_shift),
})



## Systemerweiterung bei 25 % Labels

Die folgende Stufe bringt die **starke klassische Struktur-Baseline** und die
**End-to-End-Transformer** aus der früheren Versuchslinie zurück, verwendet aber die
**jetzt eingefrorenen 40k-aware Testdefinitionen**.

Damit werden keine alten, methodisch anders definierten OOD-Zahlen mit dem FINAL FREEZE
vermischt.

- `B0_STRUCT_XGB` wird auf den expliziten Strukturmerkmalen trainiert.
- Seine XGBoost-Parameter werden **nur innerhalb des 25-%-Trainingssubsets** per 3-fold CV gewählt.
- `T0_E2E` und `DAPT_E2E` erhalten exakt dieselben Labels, dieselben Tokens und denselben
  Fine-Tuning-Ablauf.
- DAPT-40k wird **nicht neu vortrainiert**; die vorhandenen fünf DAPT-Encoder werden geladen.
- Für `BASE_EMB_MLP`, `DAPT_EMB_MLP` und `CONTRASTIVE_EMB_MLP` werden die Scores aus
  der korrigierten Freeze-Auswertung wiederverwendet.


In [ ]:
# ============================================================
# 07 – Explizite Strukturmerkmale für B0
#      robuster Loader für historische Feature-Repräsentationen
# ============================================================

# Kanonische 36 Merkmale aus der früheren B0-Versuchslinie.
# Diese Namen dienen nur der nachvollziehbaren Spaltenbezeichnung; die Werte
# werden unverändert aus den bereits gespeicherten Feature-Vektoren übernommen.
CANONICAL_B0_FEATURE_NAMES = [
    "url_length",
    "host_length",
    "path_length",
    "query_length",
    "digit_ratio_url",
    "dot_count",
    "hyphen_count",
    "at_count",
    "percent_count",
    "equals_count",
    "ampersand_count",
    "slash_count",
    "subdomain_count",
    "https_flag",
    "ip_host_flag",
    "punycode_flag",
    "host_entropy",
    "path_entropy",
    "suspicious_terms_url",
    "html_length",
    "visible_text_length",
    "title_length",
    "form_count",
    "input_count",
    "password_input_count",
    "hidden_input_count",
    "button_count",
    "link_count",
    "external_link_ratio",
    "external_form_action_count",
    "script_count",
    "iframe_count",
    "meta_refresh_count",
    "event_handler_count",
    "suspicious_terms_text",
    "text_markup_ratio",
]

if "features" not in train_df.columns:
    raise KeyError(
        "Spalte 'features' fehlt. B0_STRUCT_XGB benötigt die bereits "
        "gespeicherten URL-/HTML-Strukturmerkmale."
    )

def decode_feature_object(x):
    """
    Akzeptiert historische Speicherformen:
    - dict
    - list / tuple
    - numpy array / pandas Series
    - JSON- oder Python-literal-String eines dict/list
    """
    if x is None:
        return None

    if isinstance(x, dict):
        return x

    if isinstance(x, pd.Series):
        return x.to_numpy()

    if isinstance(x, np.ndarray):
        return x

    if isinstance(x, (list, tuple)):
        return np.asarray(x)

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return None

        # JSON zuerst.
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                return obj
            if isinstance(obj, (list, tuple)):
                return np.asarray(obj)
        except Exception:
            pass

        # Historische Pickle/DataFrame-Repräsentationen können Python-Literale
        # statt valider JSON-Syntax enthalten.
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, dict):
                return obj
            if isinstance(obj, (list, tuple)):
                return np.asarray(obj)
        except Exception:
            pass

        # Letzter Fallback für einfache "[1 2 3 ...]"/"1,2,3"-Darstellungen.
        stripped = s.strip("[]()")
        for sep in [",", " "]:
            try:
                parts = [p for p in stripped.replace("\n", " ").split(sep) if p.strip()]
                arr = np.asarray([float(p) for p in parts], dtype=np.float32)
                if arr.size:
                    return arr
            except Exception:
                continue

    return None


# Typ-Audit vor der eigentlichen Konvertierung.
sample_types = (
    train_df["features"]
    .head(min(100, len(train_df)))
    .map(lambda x: type(x).__name__)
    .value_counts()
    .to_dict()
)
print({"b0_feature_raw_types_first100": sample_types})


# Zuerst Storage-Modus anhand des ersten dekodierbaren Trainingsobjekts bestimmen.
decoded_first = None
for raw in train_df["features"]:
    obj = decode_feature_object(raw)
    if obj is not None:
        decoded_first = obj
        break

if decoded_first is None:
    raise RuntimeError(
        "Die Spalte 'features' ist vorhanden, aber kein Eintrag konnte "
        "als Strukturmerkmal dekodiert werden."
    )

if isinstance(decoded_first, dict):
    FEATURE_STORAGE_MODE = "dict"

    # Union nur auf TRAIN bestimmen -> kein Testwissen für Featureschema.
    train_dicts = []
    for raw in train_df["features"]:
        obj = decode_feature_object(raw)
        if obj is None:
            obj = {}
        if not isinstance(obj, dict):
            raise RuntimeError(
                "Inkonsistente Feature-Speicherform: Training mischt dict und Vektor."
            )
        train_dicts.append(obj)

    tmp_train = pd.json_normalize(train_dicts, sep="__")

    numeric_cols = []
    for c in tmp_train.columns:
        converted = pd.to_numeric(tmp_train[c], errors="coerce")
        if converted.notna().any():
            numeric_cols.append(c)

    if not numeric_cols:
        raise RuntimeError(
            "Feature-Dictionaries wurden gefunden, enthalten aber keine "
            "numerisch interpretierbaren Werte."
        )

    FEATURE_COLUMNS = sorted(numeric_cols)

    def feature_matrix(frame):
        decoded = []
        for raw in frame["features"]:
            obj = decode_feature_object(raw)
            if obj is None:
                obj = {}
            if not isinstance(obj, dict):
                raise RuntimeError(
                    "Inkonsistente Feature-Speicherform außerhalb des Trainings."
                )
            decoded.append(obj)

        tmp = pd.json_normalize(decoded, sep="__")
        tmp = tmp.reindex(columns=FEATURE_COLUMNS)
        for c in FEATURE_COLUMNS:
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce")
        return (
            tmp.replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
            .to_numpy(dtype=np.float32)
        )

else:
    FEATURE_STORAGE_MODE = "vector"
    first_vec = np.asarray(decoded_first).reshape(-1)
    FEATURE_DIM = int(first_vec.size)

    if FEATURE_DIM <= 0:
        raise RuntimeError("Dekodierter Feature-Vektor ist leer.")

    if FEATURE_DIM == len(CANONICAL_B0_FEATURE_NAMES):
        FEATURE_COLUMNS = list(CANONICAL_B0_FEATURE_NAMES)
    else:
        # Kein Raten: unbekannte Dimension bekommt neutrale reproduzierbare Namen.
        FEATURE_COLUMNS = [f"feature_{i:03d}" for i in range(FEATURE_DIM)]

    def feature_matrix(frame):
        rows = []
        bad = []
        for row_i, raw in enumerate(frame["features"]):
            obj = decode_feature_object(raw)
            if obj is None or isinstance(obj, dict):
                bad.append((row_i, type(obj).__name__ if obj is not None else "None"))
                continue

            arr = np.asarray(obj).reshape(-1)
            try:
                arr = pd.to_numeric(pd.Series(arr), errors="coerce").to_numpy(dtype=np.float32)
            except Exception:
                bad.append((row_i, "non_numeric"))
                continue

            if arr.size != FEATURE_DIM:
                raise RuntimeError(
                    f"Uneinheitliche B0-Feature-Dimension in Zeile {row_i}: "
                    f"{arr.size} statt {FEATURE_DIM}."
                )
            rows.append(arr)

        if bad:
            raise RuntimeError(
                f"{len(bad)} Feature-Zeilen konnten nicht konsistent dekodiert werden. "
                f"Erste Beispiele: {bad[:5]}"
            )

        X = np.vstack(rows).astype(np.float32, copy=False)
        X[~np.isfinite(X)] = np.nan

        # Fehlende Werte ausschließlich mit TRAIN-unabhängiger Konstante 0 ersetzen.
        return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)


X_STRUCT = {
    "train": feature_matrix(train_df),
    "calibration": feature_matrix(calibration_df),
    "iid": feature_matrix(iid_df),
    "holdout": feature_matrix(holdout_df),
}

# Harte Konsistenzprüfungen.
for split_name, X in X_STRUCT.items():
    expected_rows = {
        "train": len(train_df),
        "calibration": len(calibration_df),
        "iid": len(iid_df),
        "holdout": len(holdout_df),
    }[split_name]
    if X.shape != (expected_rows, len(FEATURE_COLUMNS)):
        raise RuntimeError(
            f"B0-Matrix {split_name}: {X.shape}, erwartet "
            f"({expected_rows}, {len(FEATURE_COLUMNS)})."
        )
    if not np.isfinite(X).all():
        raise RuntimeError(f"B0-Matrix {split_name} enthält nicht-finite Werte.")

pd.DataFrame({
    "feature_index": np.arange(len(FEATURE_COLUMNS)),
    "feature": FEATURE_COLUMNS,
}).to_csv(
    OUTPUT_ROOT / "b0_structural_feature_columns.csv",
    index=False,
)

print({
    "b0_feature_storage_mode": FEATURE_STORAGE_MODE,
    "b0_n_features": len(FEATURE_COLUMNS),
    "canonical_36_names_used": len(FEATURE_COLUMNS) == 36
        and FEATURE_COLUMNS == CANONICAL_B0_FEATURE_NAMES,
    "train_shape": X_STRUCT["train"].shape,
    "calibration_shape": X_STRUCT["calibration"].shape,
    "iid_shape": X_STRUCT["iid"].shape,
    "holdout_shape": X_STRUCT["holdout"].shape,
})

In [ ]:

# ============================================================
# 08 – B0_STRUCT_XGB: train-only CV bei 25 % Labels
# ============================================================

B0_CANDIDATES = [
    {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.05, "min_child_weight": 1},
    {"n_estimators": 500, "max_depth": 3, "learning_rate": 0.05, "min_child_weight": 1},
    {"n_estimators": 400, "max_depth": 4, "learning_rate": 0.10, "min_child_weight": 2},
    {"n_estimators": 400, "max_depth": 5, "learning_rate": 0.05, "min_child_weight": 1},
]

def build_b0(params, seed):
    return XGBClassifier(
        **params,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=2,
        random_state=seed,
    )

SYSTEM_SCORE_ROOT = OUTPUT_ROOT / "system_scores"
SYSTEM_SCORE_ROOT.mkdir(exist_ok=True)

def system_score_path(model_name, seed):
    return SYSTEM_SCORE_ROOT / f"{model_name}_seed{seed}.npz"

b0_tuning_rows = []

for seed in SEEDS:
    idx = BUDGET_INDICES[seed][SYSTEM_LABEL_BUDGET]
    X = X_STRUCT["train"][idx]
    y = y_train[idx]

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    best = None
    for ci, params in enumerate(B0_CANDIDATES):
        vals = []
        for fold, (tr, va) in enumerate(cv.split(X, y)):
            m = build_b0(params, seed + fold)
            m.fit(X[tr], y[tr])
            p = m.predict_proba(X[va])[:, 1]
            vals.append(average_precision_score(y[va], p))
        mean_ap = float(np.mean(vals))
        b0_tuning_rows.append({
            "seed": seed,
            "candidate": ci,
            "params": json.dumps(params, sort_keys=True),
            "cv_ap": mean_ap,
        })
        if best is None or mean_ap > best[0]:
            best = (mean_ap, params)

    score_file = system_score_path("B0_STRUCT_XGB", seed)
    if not score_file.exists():
        t0 = time.perf_counter()
        model = build_b0(best[1], seed)
        model.fit(X, y)
        fit_seconds = time.perf_counter() - t0
        np.savez_compressed(
            score_file,
            cal=model.predict_proba(X_STRUCT["calibration"])[:, 1].astype(np.float32),
            iid=model.predict_proba(X_STRUCT["iid"])[:, 1].astype(np.float32),
            holdout=model.predict_proba(X_STRUCT["holdout"])[:, 1].astype(np.float32),
            fit_seconds=np.asarray([fit_seconds]),
            params_json=np.asarray([json.dumps(best[1], sort_keys=True)]),
        )
        del model

    print({
        "b0_seed": seed,
        "n_labels": len(idx),
        "best_train_cv_ap": round(best[0], 6),
        "params": best[1],
    })

b0_tuning_df = pd.DataFrame(b0_tuning_rows)
b0_tuning_df.to_csv(OUTPUT_ROOT / "b0_train_only_tuning.csv", index=False)


In [ ]:

# ============================================================
# 09 – T0_E2E und DAPT_E2E
#      gleiche Tokens, Labels und Fine-Tuning-Prozedur
# ============================================================

class TokenSubsetDataset(Dataset):
    def __init__(self, arrays, labels, indices):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]
        self.labels = np.asarray(labels, dtype=np.int64)
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        j = int(self.indices[i])
        return (
            torch.tensor(self.ids[j], dtype=torch.long),
            torch.tensor(self.mask[j], dtype=torch.long),
            torch.tensor(self.labels[j], dtype=torch.long),
        )

class TokenAllDataset(Dataset):
    def __init__(self, arrays):
        self.ids = arrays["input_ids"]
        self.mask = arrays["attention_mask"]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return (
            torch.tensor(self.ids[i], dtype=torch.long),
            torch.tensor(self.mask[i], dtype=torch.long),
        )

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def score_e2e_model(model, arrays, batch_size=E2E_SCORE_BATCH_SIZE):
    ds = TokenAllDataset(arrays)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=torch.cuda.is_available()
    )
    model.eval()
    scores = []
    use_amp = torch.cuda.is_available()
    for ids, mask in loader:
        ids = ids.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(
            device_type="cuda", dtype=torch.float16, enabled=use_amp
        ):
            logits = model(input_ids=ids, attention_mask=mask).logits
        scores.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(scores).astype(np.float32)

def train_one_e2e(init_dir, seed, train_idx, model_name):
    score_file = system_score_path(model_name, seed)
    if score_file.exists():
        print({"e2e_reuse": model_name, "seed": seed})
        return

    y = y_train
    last_error = None

    for batch_size in E2E_BATCH_CANDIDATES:
        try:
            set_all_seeds(seed)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            model = AutoModelForSequenceClassification.from_pretrained(
                str(init_dir),
                num_labels=2,
                local_files_only=True,
                ignore_mismatched_sizes=True,
            )
            model.to(DEVICE)

            ds = TokenSubsetDataset(TOKENS["train"], y, train_idx)
            generator = torch.Generator()
            generator.manual_seed(seed)
            loader = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=True,
                generator=generator,
                num_workers=2,
                pin_memory=torch.cuda.is_available(),
            )

            # OOM-Fallback hält die effektive Batchgröße ungefähr bei 16.
            grad_accum = max(1, 16 // batch_size)
            updates_per_epoch = math.ceil(len(loader) / grad_accum)
            total_updates = updates_per_epoch * E2E_EPOCHS
            warmup_steps = int(round(total_updates * E2E_WARMUP_RATIO))

            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=E2E_LR,
                weight_decay=E2E_WEIGHT_DECAY,
            )
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_updates,
            )

            use_amp = torch.cuda.is_available()
            scaler = torch.amp.GradScaler("cuda") if use_amp else None

            history = []
            started = time.perf_counter()

            for epoch in range(E2E_EPOCHS):
                model.train()
                optimizer.zero_grad(set_to_none=True)
                losses = []

                for step, (ids, mask, labels) in enumerate(loader):
                    ids = ids.to(DEVICE, non_blocking=True)
                    mask = mask.to(DEVICE, non_blocking=True)
                    labels = labels.to(DEVICE, non_blocking=True)

                    with torch.amp.autocast(
                        device_type="cuda", dtype=torch.float16, enabled=use_amp
                    ):
                        out = model(
                            input_ids=ids,
                            attention_mask=mask,
                            labels=labels,
                        )
                        loss = out.loss / grad_accum

                    if scaler is not None:
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()

                    losses.append(float(loss.detach().cpu()) * grad_accum)

                    should_step = ((step + 1) % grad_accum == 0) or (step + 1 == len(loader))
                    if should_step:
                        if scaler is not None:
                            scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        if scaler is not None:
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            optimizer.step()
                        scheduler.step()
                        optimizer.zero_grad(set_to_none=True)

                history.append({
                    "model": model_name,
                    "seed": seed,
                    "epoch": epoch + 1,
                    "train_loss": float(np.mean(losses)),
                    "batch_size": batch_size,
                    "gradient_accumulation": grad_accum,
                })
                print(history[-1])

            fit_seconds = time.perf_counter() - started

            cal_score = score_e2e_model(model, TOKENS["calibration"])
            iid_score = score_e2e_model(model, TOKENS["iid"])
            hold_score = score_e2e_model(model, TOKENS["holdout"])

            np.savez_compressed(
                score_file,
                cal=cal_score,
                iid=iid_score,
                holdout=hold_score,
                fit_seconds=np.asarray([fit_seconds]),
                batch_size=np.asarray([batch_size]),
                grad_accum=np.asarray([grad_accum]),
            )

            hist_path = OUTPUT_ROOT / f"{model_name}_training_history_seed{seed}.csv"
            pd.DataFrame(history).to_csv(hist_path, index=False)

            del model, optimizer, scheduler, loader, ds
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return

        except RuntimeError as exc:
            last_error = exc
            if "out of memory" not in str(exc).lower():
                raise
            print({
                "oom_retry": model_name,
                "seed": seed,
                "failed_batch_size": batch_size,
            })
            try:
                del model
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"{model_name} Seed {seed}: alle Batchgrößen fehlgeschlagen."
    ) from last_error

for seed in SEEDS:
    train_idx = BUDGET_INDICES[seed][SYSTEM_LABEL_BUDGET]

    train_one_e2e(
        BASE_MODEL_DIR,
        seed,
        train_idx,
        "T0_E2E",
    )
    train_one_e2e(
        DAPT_ENCODERS[seed],
        seed,
        train_idx,
        "DAPT_E2E",
    )


In [ ]:

# ============================================================
# 10 – Embedding-Systemkandidaten aus dem FINAL FREEZE übernehmen
# ============================================================

EMBED_SYSTEMS = {
    "BASE_EMB_MLP": ("BASE", "MLP"),
    "DAPT_EMB_MLP": ("DAPT", "MLP"),
    "CONTRASTIVE_EMB_MLP": ("CONTRASTIVE", "MLP"),
}

for seed in SEEDS:
    for system_name, (rep, kind) in EMBED_SYSTEMS.items():
        src = freeze_score_path(rep, kind, seed, SYSTEM_LABEL_BUDGET)
        if not src.exists():
            raise FileNotFoundError(src)
        dat = np.load(src)
        dst = system_score_path(system_name, seed)
        if not dst.exists():
            np.savez_compressed(
                dst,
                cal=dat["cal"].astype(np.float32),
                iid=dat["iid"].astype(np.float32),
                holdout=dat["holdout"].astype(np.float32),
                fit_seconds=dat["fit_seconds"] if "fit_seconds" in dat.files else np.asarray([np.nan]),
            )

print({"embedding_system_scores": "ready"})


In [ ]:

# ============================================================
# 11 – Systemvergleich bei 25 % Labels, FPR überall
# ============================================================

SYSTEM_MODELS = [
    "B0_STRUCT_XGB",
    "T0_E2E",
    "DAPT_E2E",
    "BASE_EMB_MLP",
    "DAPT_EMB_MLP",
    "CONTRASTIVE_EMB_MLP",
]

system_rows = []

for seed in SEEDS:
    ycal = calibration_df["label"].to_numpy(dtype=int)

    for model_name in SYSTEM_MODELS:
        dat = np.load(system_score_path(model_name, seed))
        cal_score = dat["cal"]
        iid_score = dat["iid"]
        hold_score = dat["holdout"]
        fit_seconds = float(dat["fit_seconds"][0]) if "fit_seconds" in dat.files else np.nan
        ops = calibration_operating_points(ycal, cal_score)

        for target in TARGET_FPRS:
            op = ops[target]
            for scenario in SCENARIOS:
                yev, sev = scenario_score_view(iid_score, hold_score, scenario)
                system_rows.append({
                    "model": model_name,
                    "seed": seed,
                    "label_budget": SYSTEM_LABEL_BUDGET,
                    "n_train": len(BUDGET_INDICES[seed][SYSTEM_LABEL_BUDGET]),
                    "scenario": scenario,
                    "n_test": len(yev),
                    "target_fpr": target,
                    "threshold": op["threshold"],
                    "calibration_empirical_fpr": op["calibration_empirical_fpr"],
                    "calibration_recall": op["calibration_recall"],
                    "fit_seconds": fit_seconds,
                    **metric_row(yev, sev, op["threshold"]),
                })

system_results = pd.DataFrame(system_rows)
system_results.to_csv(OUTPUT_ROOT / "system_25pct_results.csv", index=False)

system_summary = (
    system_results
    .groupby(["model", "scenario", "target_fpr"], as_index=False)
    .agg(
        n_test=("n_test", "first"),
        prevalence_phish=("prevalence_phish", "first"),
        average_precision_mean=("average_precision", "mean"),
        average_precision_std=("average_precision", "std"),
        roc_auc_mean=("roc_auc", "mean"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=("precision", "mean"),
        f1_mean=("f1", "mean"),
        calibration_empirical_fpr_mean=("calibration_empirical_fpr", "mean"),
        empirical_fpr_mean=("empirical_fpr", "mean"),
        empirical_fpr_std=("empirical_fpr", "std"),
        fp_mean=("fp", "mean"),
        fn_mean=("fn", "mean"),
    )
)
system_summary.to_csv(OUTPUT_ROOT / "system_25pct_summary.csv", index=False)
display(system_summary)


In [ ]:

# ============================================================
# 12 – Korrigierter AP-Shift auf Systemebene
# ============================================================

system_ap_shift_rows = []

for keys, g in system_results.groupby(["model", "seed", "target_fpr"]):
    iid = g[g["scenario"] == "IID_BALANCED"].iloc[0]
    for scenario in ["TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"]:
        ood = g[g["scenario"] == scenario].iloc[0]
        system_ap_shift_rows.append({
            "model": keys[0],
            "seed": keys[1],
            "target_fpr": keys[2],
            "comparison": f"IID_BALANCED->{scenario}",
            "iid_n": int(iid["n_test"]),
            "ood_n": int(ood["n_test"]),
            "iid_prevalence": float(iid["prevalence_phish"]),
            "ood_prevalence": float(ood["prevalence_phish"]),
            "iid_ap": float(iid["average_precision"]),
            "ood_ap": float(ood["average_precision"]),
            "ap_delta_ood_minus_iid": float(ood["average_precision"] - iid["average_precision"]),
            "iid_empirical_fpr": float(iid["empirical_fpr"]),
            "ood_empirical_fpr": float(ood["empirical_fpr"]),
            "empirical_fpr_delta": float(ood["empirical_fpr"] - iid["empirical_fpr"]),
            "iid_recall": float(iid["recall"]),
            "ood_recall": float(ood["recall"]),
            "recall_delta": float(ood["recall"] - iid["recall"]),
        })

system_ap_shift = pd.DataFrame(system_ap_shift_rows)
system_ap_shift.to_csv(OUTPUT_ROOT / "system_ap_shift_corrected.csv", index=False)


In [ ]:

# ============================================================
# 13 – Kaskade: B0 blockiert, Stage 2 priorisiert Review
# ============================================================

def scores_for_system(model_name, seed):
    dat = np.load(system_score_path(model_name, seed))
    return dat["cal"], dat["iid"], dat["holdout"]

cascade_rankers = [
    "B0_SELF",
    "T0_E2E",
    "DAPT_E2E",
    "BASE_EMB_MLP",
    "DAPT_EMB_MLP",
    "CONTRASTIVE_EMB_MLP",
]

cascade_rows = []

for seed in SEEDS:
    ycal = calibration_df["label"].to_numpy(dtype=int)

    b0_cal, b0_iid, b0_hold = scores_for_system("B0_STRUCT_XGB", seed)
    b0_ops = calibration_operating_points(ycal, b0_cal)

    ranker_cache = {}
    ranker_ops = {}
    for ranker in cascade_rankers:
        source_model = "B0_STRUCT_XGB" if ranker == "B0_SELF" else ranker
        cal_s, iid_s, hold_s = scores_for_system(source_model, seed)
        ranker_cache[ranker] = (iid_s, hold_s)
        ranker_ops[ranker] = calibration_operating_points(ycal, cal_s)

    for target in TARGET_FPRS:
        b0_op = b0_ops[target]

        for scenario in SCENARIOS:
            y, b0_score = scenario_score_view(b0_iid, b0_hold, scenario)
            b0_pred = b0_score >= b0_op["threshold"]
            stage1_negative = ~b0_pred

            tn, fp, fn, tp = confusion_matrix(y, b0_pred.astype(int), labels=[0, 1]).ravel()
            stage1_empirical_fpr = fp / max(fp + tn, 1)
            stage1_recall = tp / max(tp + fn, 1)
            base_fn_mask = (y == 1) & stage1_negative
            n_base_fn = int(base_fn_mask.sum())
            neg_idx = np.flatnonzero(stage1_negative)

            for review_frac in REVIEW_FRACTIONS:
                n_review = max(1, int(math.ceil(review_frac * len(neg_idx))))

                for ranker in cascade_rankers:
                    riid, rhold = ranker_cache[ranker]
                    _, rank_score = scenario_score_view(riid, rhold, scenario)

                    ordered = neg_idx[np.argsort(rank_score[neg_idx])[::-1]]
                    review_idx = ordered[:n_review]

                    phish_review = int((y[review_idx] == 1).sum())
                    benign_review = int((y[review_idx] == 0).sum())
                    rescued = int(base_fn_mask[review_idx].sum())

                    # Eigener FPR-Arbeitspunkt des Rankers wird zusätzlich ausgewiesen.
                    ranker_op = ranker_ops[ranker][target]
                    ranker_pred = rank_score >= ranker_op["threshold"]
                    r_tn, r_fp, r_fn, r_tp = confusion_matrix(
                        y, ranker_pred.astype(int), labels=[0, 1]
                    ).ravel()
                    ranker_empirical_fpr = r_fp / max(r_fp + r_tn, 1)

                    review_assisted_recall_upper = (
                        (tp + rescued) / max(tp + fn, 1)
                    )

                    cascade_rows.append({
                        "seed": seed,
                        "scenario": scenario,
                        "target_fpr": target,

                        "stage1_model": "B0_STRUCT_XGB",
                        "stage1_threshold": b0_op["threshold"],
                        "stage1_calibration_empirical_fpr": b0_op["calibration_empirical_fpr"],
                        "stage1_empirical_fpr": stage1_empirical_fpr,
                        "stage1_fp": int(fp),
                        "stage1_tn": int(tn),
                        "stage1_tp": int(tp),
                        "stage1_fn": int(fn),
                        "stage1_recall": stage1_recall,

                        "ranker": ranker,
                        "ranker_own_threshold": ranker_op["threshold"],
                        "ranker_calibration_empirical_fpr": ranker_op["calibration_empirical_fpr"],
                        "ranker_empirical_fpr_at_own_threshold": ranker_empirical_fpr,

                        "review_fraction_of_stage1_negatives": review_frac,
                        "stage1_negative_n": int(len(neg_idx)),
                        "review_n": int(n_review),
                        "review_fraction_of_all_cases": float(n_review / len(y)),
                        "review_phishing_n": phish_review,
                        "review_benign_n": benign_review,
                        "review_precision": float(phish_review / max(n_review, 1)),
                        "rescued_stage1_fn": rescued,
                        "capture_rate_of_stage1_fn": float(rescued / max(n_base_fn, 1)),
                        "review_assisted_recall_upper_bound": float(review_assisted_recall_upper),
                        "potential_recall_gain_pp": float(
                            100 * (review_assisted_recall_upper - stage1_recall)
                        ),
                    })

cascade = pd.DataFrame(cascade_rows)

# Lift gegenüber B0-Self bei exakt identischer Queuegröße.
self_ref = (
    cascade[cascade["ranker"] == "B0_SELF"][
        ["seed", "scenario", "target_fpr", "review_fraction_of_stage1_negatives",
         "rescued_stage1_fn", "review_precision",
         "review_assisted_recall_upper_bound"]
    ]
    .rename(columns={
        "rescued_stage1_fn": "self_rescued_stage1_fn",
        "review_precision": "self_review_precision",
        "review_assisted_recall_upper_bound": "self_review_assisted_recall_upper_bound",
    })
)

cascade = cascade.merge(
    self_ref,
    on=["seed", "scenario", "target_fpr", "review_fraction_of_stage1_negatives"],
    how="left",
)
cascade["rescued_fn_lift_vs_self"] = (
    cascade["rescued_stage1_fn"] - cascade["self_rescued_stage1_fn"]
)
cascade["review_precision_lift_vs_self"] = (
    cascade["review_precision"] - cascade["self_review_precision"]
)
cascade["recall_upper_bound_lift_vs_self_pp"] = 100 * (
    cascade["review_assisted_recall_upper_bound"]
    - cascade["self_review_assisted_recall_upper_bound"]
)

cascade.to_csv(OUTPUT_ROOT / "cascade_results.csv", index=False)

cascade_summary = (
    cascade
    .groupby(
        ["scenario", "target_fpr", "ranker", "review_fraction_of_stage1_negatives"],
        as_index=False
    )
    .agg(
        stage1_empirical_fpr_mean=("stage1_empirical_fpr", "mean"),
        ranker_empirical_fpr_mean=("ranker_empirical_fpr_at_own_threshold", "mean"),
        stage1_recall_mean=("stage1_recall", "mean"),
        review_precision_mean=("review_precision", "mean"),
        rescued_stage1_fn_mean=("rescued_stage1_fn", "mean"),
        capture_rate_of_stage1_fn_mean=("capture_rate_of_stage1_fn", "mean"),
        review_assisted_recall_upper_bound_mean=("review_assisted_recall_upper_bound", "mean"),
        potential_recall_gain_pp_mean=("potential_recall_gain_pp", "mean"),
        rescued_fn_lift_vs_self_mean=("rescued_fn_lift_vs_self", "mean"),
        recall_upper_bound_lift_vs_self_pp_mean=("recall_upper_bound_lift_vs_self_pp", "mean"),
    )
)
cascade_summary.to_csv(OUTPUT_ROOT / "cascade_summary.csv", index=False)
display(cascade_summary.head(40))


In [ ]:

# ============================================================
# 14 – Fehlerkomplementarität: B0 gegen alle relevanten Modelle
# ============================================================

PAIR_MODELS = [
    "T0_E2E",
    "DAPT_E2E",
    "BASE_EMB_MLP",
    "DAPT_EMB_MLP",
    "CONTRASTIVE_EMB_MLP",
]

error_rows = []

for seed in SEEDS:
    ycal = calibration_df["label"].to_numpy(dtype=int)

    b0_cal, b0_iid, b0_hold = scores_for_system("B0_STRUCT_XGB", seed)
    b0_ops = calibration_operating_points(ycal, b0_cal)

    for candidate in PAIR_MODELS:
        c_cal, c_iid, c_hold = scores_for_system(candidate, seed)
        c_ops = calibration_operating_points(ycal, c_cal)

        for target in TARGET_FPRS:
            b0_op = b0_ops[target]
            c_op = c_ops[target]

            for scenario in SCENARIOS:
                y, b0_score = scenario_score_view(b0_iid, b0_hold, scenario)
                _, c_score = scenario_score_view(c_iid, c_hold, scenario)

                bp = b0_score >= b0_op["threshold"]
                cp = c_score >= c_op["threshold"]

                b_fn = (y == 1) & (~bp)
                c_fn = (y == 1) & (~cp)
                b_fp = (y == 0) & bp
                c_fp = (y == 0) & cp

                b_tn = (y == 0) & (~bp)
                c_tn = (y == 0) & (~cp)

                common_fn = int((b_fn & c_fn).sum())
                rescued_b0_fn = int((b_fn & (~c_fn)).sum())
                regressed_b0_tp = int(((y == 1) & bp & c_fn).sum())

                common_fp = int((b_fp & c_fp).sum())
                avoided_b0_fp = int((b_fp & (~c_fp)).sum())
                new_fp_vs_b0 = int((b_tn & c_fp).sum())

                b0_fpr = float(b_fp.sum() / max((y == 0).sum(), 1))
                cand_fpr = float(c_fp.sum() / max((y == 0).sum(), 1))

                fn_union = int((b_fn | c_fn).sum())
                fp_union = int((b_fp | c_fp).sum())

                error_rows.append({
                    "seed": seed,
                    "scenario": scenario,
                    "target_fpr": target,
                    "base_model": "B0_STRUCT_XGB",
                    "candidate_model": candidate,

                    "base_calibration_empirical_fpr": b0_op["calibration_empirical_fpr"],
                    "candidate_calibration_empirical_fpr": c_op["calibration_empirical_fpr"],
                    "base_empirical_fpr": b0_fpr,
                    "candidate_empirical_fpr": cand_fpr,
                    "candidate_minus_base_fpr": cand_fpr - b0_fpr,

                    "base_fn": int(b_fn.sum()),
                    "candidate_fn": int(c_fn.sum()),
                    "common_fn": common_fn,
                    "rescued_b0_fn": rescued_b0_fn,
                    "regressed_b0_tp": regressed_b0_tp,
                    "fn_jaccard": float(common_fn / max(fn_union, 1)),
                    "fn_rescue_rate": float(rescued_b0_fn / max(b_fn.sum(), 1)),

                    "base_fp": int(b_fp.sum()),
                    "candidate_fp": int(c_fp.sum()),
                    "common_fp": common_fp,
                    "avoided_b0_fp": avoided_b0_fp,
                    "new_fp_vs_b0": new_fp_vs_b0,
                    "fp_jaccard": float(common_fp / max(fp_union, 1)),
                })

error_df = pd.DataFrame(error_rows)
error_df.to_csv(OUTPUT_ROOT / "error_complementarity.csv", index=False)

error_summary = (
    error_df
    .groupby(["scenario", "target_fpr", "candidate_model"], as_index=False)
    .agg(
        base_empirical_fpr_mean=("base_empirical_fpr", "mean"),
        candidate_empirical_fpr_mean=("candidate_empirical_fpr", "mean"),
        candidate_minus_base_fpr_mean=("candidate_minus_base_fpr", "mean"),
        base_fn_mean=("base_fn", "mean"),
        candidate_fn_mean=("candidate_fn", "mean"),
        rescued_b0_fn_mean=("rescued_b0_fn", "mean"),
        regressed_b0_tp_mean=("regressed_b0_tp", "mean"),
        fn_rescue_rate_mean=("fn_rescue_rate", "mean"),
        fn_jaccard_mean=("fn_jaccard", "mean"),
        base_fp_mean=("base_fp", "mean"),
        candidate_fp_mean=("candidate_fp", "mean"),
        new_fp_vs_b0_mean=("new_fp_vs_b0", "mean"),
        avoided_b0_fp_mean=("avoided_b0_fp", "mean"),
        fp_jaccard_mean=("fp_jaccard", "mean"),
    )
)
error_summary.to_csv(OUTPUT_ROOT / "error_complementarity_summary.csv", index=False)
display(error_summary.head(40))


In [ ]:

# ============================================================
# 15 – Fehlerprofile und qualitative Beispiele (Seed 42)
# ============================================================

def enrich_frame(frame):
    f = frame.reset_index(drop=True).copy()
    f["text_chars"] = f["text"].fillna("").astype(str).str.len()
    f["url_chars"] = (
        f["url"].fillna("").astype(str).str.len()
        if "url" in f.columns else np.nan
    )
    return f

feature_rows = []
sample_rows = []
seed = 42
ycal = calibration_df["label"].to_numpy(dtype=int)

b0_cal, b0_iid, b0_hold = scores_for_system("B0_STRUCT_XGB", seed)
b0_ops = calibration_operating_points(ycal, b0_cal)

for candidate in ["T0_E2E", "DAPT_E2E", "DAPT_EMB_MLP", "CONTRASTIVE_EMB_MLP"]:
    c_cal, c_iid, c_hold = scores_for_system(candidate, seed)
    c_ops = calibration_operating_points(ycal, c_cal)

    for target in TARGET_FPRS:
        for scenario in SCENARIOS:
            source, idx = SCENARIOS[scenario]
            frame = iid_df.iloc[idx] if source == "iid" else holdout_df.iloc[idx]
            frame = enrich_frame(frame)

            y, b0_score = scenario_score_view(b0_iid, b0_hold, scenario)
            _, c_score = scenario_score_view(c_iid, c_hold, scenario)

            bp = b0_score >= b0_ops[target]["threshold"]
            cp = c_score >= c_ops[target]["threshold"]

            base_fn = (y == 1) & (~bp)
            rescued = base_fn & cp
            common_fn = base_fn & (~cp)

            for group_name, mask in [
                ("B0_FN_RESCUED_BY_CANDIDATE", rescued),
                ("COMMON_FN", common_fn),
            ]:
                pos = np.flatnonzero(mask)
                if len(pos):
                    for feature in ["text_chars", "url_chars"]:
                        vals = pd.to_numeric(frame.iloc[pos][feature], errors="coerce")
                        if vals.notna().any():
                            feature_rows.append({
                                "seed": seed,
                                "candidate_model": candidate,
                                "scenario": scenario,
                                "target_fpr": target,
                                "group": group_name,
                                "feature": feature,
                                "n": int(vals.notna().sum()),
                                "mean": float(vals.mean()),
                                "median": float(vals.median()),
                                "q25": float(vals.quantile(0.25)),
                                "q75": float(vals.quantile(0.75)),
                            })

            # Je Kombination maximal fünf repräsentative gerettete B0-FN.
            rescued_idx = np.flatnonzero(rescued)
            if len(rescued_idx):
                # Kandidatenscore absteigend: besonders klare Rescue-Fälle.
                rescued_idx = rescued_idx[np.argsort(c_score[rescued_idx])[::-1]][:5]
                for i in rescued_idx:
                    r = frame.iloc[int(i)]
                    sample_rows.append({
                        "seed": seed,
                        "candidate_model": candidate,
                        "scenario": scenario,
                        "target_fpr": target,
                        "base_empirical_fpr": float(
                            ((y == 0) & bp).sum() / max((y == 0).sum(), 1)
                        ),
                        "candidate_empirical_fpr": float(
                            ((y == 0) & cp).sum() / max((y == 0).sum(), 1)
                        ),
                        "b0_score": float(b0_score[i]),
                        "candidate_score": float(c_score[i]),
                        "sha256": str(r.get("sha256", "")),
                        "domain": str(r.get("domain", "")),
                        "url": str(r.get("url", ""))[:500],
                        "template_hash": str(r.get("template_hash", "")),
                        "near_duplicate_to_development": r.get(
                            "near_duplicate_to_development", np.nan
                        ),
                        "min_simhash_distance_to_development": r.get(
                            "min_simhash_distance_to_development", np.nan
                        ),
                        "text_preview": str(r.get("text", ""))[:400].replace("\n", " "),
                    })

pd.DataFrame(feature_rows).to_csv(
    OUTPUT_ROOT / "error_feature_profiles_seed42.csv", index=False
)
pd.DataFrame(sample_rows).to_csv(
    OUTPUT_ROOT / "rescued_b0_false_negatives_seed42.csv", index=False
)

print({
    "feature_profile_rows": len(feature_rows),
    "qualitative_rescue_rows": len(sample_rows),
})


In [ ]:

# ============================================================
# 16 – Kernübersichten für die spätere Bachelorarbeit
# ============================================================

# A) FINAL FREEZE: 25 % Labels, primärer 0,5%-FPR-Punkt
freeze_25_primary = freeze_summary[
    (freeze_summary["label_budget"] == 0.25)
    & (freeze_summary["target_fpr"] == PRIMARY_TARGET_FPR)
].copy()
freeze_25_primary.to_csv(
    OUTPUT_ROOT / "TABLE_freeze_25pct_primary_fpr.csv", index=False
)

# B) Systemvergleich: alle drei FPR-Punkte, Stressszenarien
system_stress_table = system_summary[
    system_summary["scenario"].isin([
        "TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
    ])
].copy()
system_stress_table.to_csv(
    OUTPUT_ROOT / "TABLE_system_stress_all_fpr.csv", index=False
)

# C) Kaskade: Stressszenarien, alle drei FPR-Punkte
cascade_stress = cascade_summary[
    cascade_summary["scenario"].isin([
        "TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
    ])
].copy()
cascade_stress.to_csv(
    OUTPUT_ROOT / "TABLE_cascade_stress_all_fpr.csv", index=False
)

# D) Fehleranalyse: Stressszenarien
error_stress = error_summary[
    error_summary["scenario"].isin([
        "TEMPORAL", "DOMAIN_OOD", "TEMPLATE_OOD", "DOMAIN_TEMPLATE_OOD"
    ])
].copy()
error_stress.to_csv(
    OUTPUT_ROOT / "TABLE_error_complementarity_stress.csv", index=False
)

print("Bachelor-Tabellen exportiert.")


In [ ]:

# ============================================================
# 17 – Completion Audit und ZIP
# ============================================================

# Erwartete korrigierte Freeze-Zeilen:
# 3 Reps × 3 Classifier × 5 Seeds × 3 Budgets × 6 Szenarien × 3 FPR
expected_freeze_rows = (
    len(REPRESENTATIONS) * len(CLASSIFIERS) * len(SEEDS)
    * len(LABEL_BUDGETS) * len(SCENARIOS) * len(TARGET_FPRS)
)
if len(freeze_results) != expected_freeze_rows:
    raise RuntimeError(
        f"freeze_corrected_results unvollständig: {len(freeze_results)} / "
        f"{expected_freeze_rows}"
    )

expected_system_rows = (
    len(SYSTEM_MODELS) * len(SEEDS) * len(SCENARIOS) * len(TARGET_FPRS)
)
if len(system_results) != expected_system_rows:
    raise RuntimeError(
        f"system_results unvollständig: {len(system_results)} / {expected_system_rows}"
    )

config = {
    "status": "COMPLETE",
    "purpose": "FINAL FREEZE extension: FPR + AP correction + 25pct systems/cascade/error analysis",
    "seeds": SEEDS,
    "label_budgets": LABEL_BUDGETS,
    "system_label_budget": SYSTEM_LABEL_BUDGET,
    "target_fprs": TARGET_FPRS,
    "primary_target_fpr": PRIMARY_TARGET_FPR,
    "scenarios": list(SCENARIOS.keys()),
    "ap_primary_reference": "IID_BALANCED (50/50 prevalence matched)",
    "ap_equal_n_sensitivity_n": int(2 * n_each_iid),
    "system_models": SYSTEM_MODELS,
    "e2e": {
        "epochs": E2E_EPOCHS,
        "lr": E2E_LR,
        "weight_decay": E2E_WEIGHT_DECAY,
        "warmup_ratio": E2E_WARMUP_RATIO,
        "max_length": MAX_LENGTH,
        "dapt_pretraining_repeated": False,
    },
    "holdout_used_for_training": False,
    "holdout_used_for_threshold_selection": False,
}

(OUTPUT_ROOT / "HYBRID_FREEZE_EXTENSION_COMPLETE.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)

archive = shutil.make_archive(
    "/kaggle/working/phreshphish_hybrid_freeze_extension",
    "zip",
    root_dir=OUTPUT_ROOT,
)

print(json.dumps(config, indent=2))
print({"zip": archive})



## Zentrale Ergebnisdateien

### AP-/FPR-Korrektur des FINAL FREEZE
- `testset_prevalence_audit.csv`
- `freeze_corrected_results.csv`
- `freeze_corrected_summary.csv`
- `ap_shift_corrected_prevalence_matched.csv`
- `ap_equal_n_sensitivity_results.csv`
- `ap_shift_equal_n_sensitivity.csv`

### Systemvergleich bei 25 % Labels
- `system_25pct_results.csv`
- `system_25pct_summary.csv`
- `system_ap_shift_corrected.csv`
- `b0_train_only_tuning.csv`

### Kaskade
- `cascade_results.csv`
- `cascade_summary.csv`

### Fehleranalyse
- `error_complementarity.csv`
- `error_complementarity_summary.csv`
- `error_feature_profiles_seed42.csv`
- `rescued_b0_false_negatives_seed42.csv`

### Direkt für Kapitel 5
- `TABLE_freeze_25pct_primary_fpr.csv`
- `TABLE_system_stress_all_fpr.csv`
- `TABLE_cascade_stress_all_fpr.csv`
- `TABLE_error_complementarity_stress.csv`

### Abschluss
- `HYBRID_FREEZE_EXTENSION_COMPLETE.json`
- `/kaggle/working/phreshphish_hybrid_freeze_extension.zip`

**Interpretationsregel:**  
`IID_ORIGINAL` bleibt für operative Kennzahlen erhalten. Für Aussagen wie
„AP verschlechtert sich unter OOD um …“ wird nur `IID_BALANCED` als Referenz
verwendet. Die Equal-N-Auswertung ist eine zusätzliche Sensitivitätsprüfung.


# FINAL BACHELOR RUN

Die bisherigen Zellen bleiben als reproduzierbare Vorstufe bestehen. Ab hier wird die
Evaluation **vollständig zusammengeführt**.

## Erweiterungen gegenüber dem vorherigen Lauf

1. B0, T0-End-to-End und DAPT-End-to-End werden bei **10 %, 25 % und 100 % Labels**
   ausgewertet.
2. Alle bisherigen beibehaltenen Lösungen werden in einer gemeinsamen Master-Tabelle
   zusammengeführt:
   - B0 Structure-XGB
   - T0 End-to-End
   - DAPT End-to-End
   - BASE / DAPT / CONTRASTIVE jeweils mit LR / XGBoost / MLP
3. 0,5 %, 1 % und 2 % Ziel-FPR werden überall ausgewiesen.
4. AP wird ausschließlich mit prävalenzvergleichbarem `IID_BALANCED` interpretiert;
   Equal-N bleibt als Sensitivitätsanalyse erhalten.
5. Die Statistik verwendet **alle fünf vorab festgelegten Seeds**. Kein Seed wird aufgrund
   seines Ergebnisses entfernt.
6. Kaskade und Fehleranalyse werden auf alle drei Labelbudgets erweitert.
7. Active Learning wird als **explorative, vom SSL-Kern getrennte Erweiterung** ergänzt:
   pool-basiertes Unsicherheits-Sampling gegen Random Sampling von 10 % bis 25 % Labels.

### Statistischer Hinweis

Bei fünf gepaarten Seeds beträgt die kleinste mögliche zweiseitige p-Ausprägung eines
exakten Sign-Flip-Tests 0,0625. Das Notebook berichtet deshalb neben dem p-Wert
insbesondere Effektgröße, Konfidenzintervall und Win/Tie/Loss über alle Seeds, statt
einzelne ungünstige Seeds nachträglich auszuschließen.

### Methodische Referenzen für die Zusatzanalysen

- Schröder, Niekler und Potthast (2022): *Revisiting Uncertainty-based Query Strategies
  for Active Learning with Transformers*, Findings of ACL 2022.
- Bosma et al. (2023): *Reproducibility of Training Deep Learning Models for Medical
  Image Analysis*, Proceedings of Machine Learning Research 227.

In [ ]:
# ============================================================
# FINAL 01 – Konfiguration und Ausgabeordner
# ============================================================
from itertools import product
from scipy.stats import t as student_t
import matplotlib.pyplot as plt

FINAL_ROOT = Path("/kaggle/working/phreshphish_FINAL_BACHELOR_RUN")
FINAL_ROOT.mkdir(parents=True, exist_ok=True)

FINAL_SCORE_ROOT = FINAL_ROOT / "system_scores"
FINAL_SCORE_ROOT.mkdir(exist_ok=True)

FINAL_TABLE_ROOT = FINAL_ROOT / "thesis_tables"
FINAL_TABLE_ROOT.mkdir(exist_ok=True)

FINAL_FIG_ROOT = FINAL_ROOT / "figures"
FINAL_FIG_ROOT.mkdir(exist_ok=True)

FINAL_AUDIT_ROOT = FINAL_ROOT / "audit"
FINAL_AUDIT_ROOT.mkdir(exist_ok=True)

FINAL_CORE_SYSTEMS = [
    "B0_STRUCT_XGB",
    "T0_E2E",
    "DAPT_E2E",
]

FINAL_EMBED_SYSTEMS = [
    f"{rep}_{clf}"
    for rep in REPRESENTATIONS
    for clf in CLASSIFIERS
]

FINAL_ALL_SYSTEMS = FINAL_CORE_SYSTEMS + FINAL_EMBED_SYSTEMS

FINAL_STRESS = [
    "TEMPORAL",
    "DOMAIN_OOD",
    "TEMPLATE_OOD",
    "DOMAIN_TEMPLATE_OOD",
]

FINAL_REVIEW_FRACTIONS = [0.05, 0.10, 0.20]

AL_BUDGETS = [0.10, 0.15, 0.20, 0.25]
AL_STRATEGIES = ["UNCERTAINTY", "RANDOM"]
AL_ACQUIRE_N = 200

def budget_tag(frac):
    return str(float(frac)).replace(".", "p")

def final_system_score_path(model, seed, frac):
    return (
        FINAL_SCORE_ROOT
        / f"{model}_seed{seed}_budget{budget_tag(frac)}.npz"
    )

print({
    "final_systems": FINAL_ALL_SYSTEMS,
    "label_budgets": LABEL_BUDGETS,
    "target_fprs": TARGET_FPRS,
    "active_learning_budgets": AL_BUDGETS,
})

In [ ]:
# ============================================================
# FINAL 02 – Erfolgreiche 25-%-Scores übernehmen
# ============================================================
# Die erfolgreiche Hybrid-Extension liegt im alten OUTPUT_ROOT/system_scores.
# Diese Dateien entsprechen exakt demselben 25-%-Subset und werden in den
# budgetierten FINAL-Score-Namensraum kopiert.

for model in FINAL_CORE_SYSTEMS:
    for seed in SEEDS:
        old = system_score_path(model, seed)
        new = final_system_score_path(model, seed, 0.25)
        if old.exists() and not new.exists():
            shutil.copy2(old, new)

print({
    "reused_25pct_core_scores": sum(
        final_system_score_path(m, s, 0.25).exists()
        for m in FINAL_CORE_SYSTEMS
        for s in SEEDS
    ),
    "expected": len(FINAL_CORE_SYSTEMS) * len(SEEDS),
})

In [ ]:
# ============================================================
# FINAL 03 – B0 Structure-XGB bei 10 / 25 / 100 %
# ============================================================

final_b0_tuning_rows = []

for seed in SEEDS:
    for frac in LABEL_BUDGETS:
        score_file = final_system_score_path(
            "B0_STRUCT_XGB", seed, frac
        )

        idx = BUDGET_INDICES[seed][frac]
        X = X_STRUCT["train"][idx]
        y = y_train[idx]

        if score_file.exists():
            final_b0_tuning_rows.append({
                "seed": seed,
                "label_budget": frac,
                "status": "REUSE",
            })
            continue

        cv = StratifiedKFold(
            n_splits=3,
            shuffle=True,
            random_state=seed,
        )

        best = None
        for ci, params in enumerate(B0_CANDIDATES):
            fold_scores = []
            for fold, (tr, va) in enumerate(cv.split(X, y)):
                m = build_b0(params, seed + fold)
                m.fit(X[tr], y[tr])
                p = m.predict_proba(X[va])[:, 1]
                fold_scores.append(
                    average_precision_score(y[va], p)
                )

            mean_ap = float(np.mean(fold_scores))
            final_b0_tuning_rows.append({
                "seed": seed,
                "label_budget": frac,
                "candidate": ci,
                "status": "TUNE",
                "cv_ap": mean_ap,
                "params": json.dumps(params, sort_keys=True),
            })

            if best is None or mean_ap > best[0]:
                best = (mean_ap, params)

        t0 = time.perf_counter()
        model = build_b0(best[1], seed)
        model.fit(X, y)
        fit_seconds = time.perf_counter() - t0

        np.savez_compressed(
            score_file,
            cal=model.predict_proba(
                X_STRUCT["calibration"]
            )[:, 1].astype(np.float32),
            iid=model.predict_proba(
                X_STRUCT["iid"]
            )[:, 1].astype(np.float32),
            holdout=model.predict_proba(
                X_STRUCT["holdout"]
            )[:, 1].astype(np.float32),
            fit_seconds=np.asarray([fit_seconds]),
            params_json=np.asarray([
                json.dumps(best[1], sort_keys=True)
            ]),
        )

        print({
            "B0_FINAL": [seed, frac],
            "best_cv_ap": round(best[0], 6),
            "params": best[1],
        })

        del model
        gc.collect()

pd.DataFrame(final_b0_tuning_rows).to_csv(
    FINAL_AUDIT_ROOT / "b0_tuning_all_budgets.csv",
    index=False,
)

In [ ]:
# ============================================================
# FINAL 04 – T0-E2E und DAPT-E2E bei 10 / 25 / 100 %
# ============================================================

def train_final_e2e(init_dir, seed, frac, model_name):
    score_file = final_system_score_path(
        model_name, seed, frac
    )

    if score_file.exists():
        print({
            "E2E_FINAL": model_name,
            "seed": seed,
            "budget": frac,
            "status": "REUSE",
        })
        return

    train_idx = BUDGET_INDICES[seed][frac]
    last_error = None

    for batch_size in E2E_BATCH_CANDIDATES:
        try:
            set_all_seeds(seed)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            model = AutoModelForSequenceClassification.from_pretrained(
                str(init_dir),
                num_labels=2,
                local_files_only=True,
                ignore_mismatched_sizes=True,
            ).to(DEVICE)

            ds = TokenSubsetDataset(
                TOKENS["train"],
                y_train,
                train_idx,
            )

            generator = torch.Generator()
            generator.manual_seed(seed)

            loader = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=True,
                generator=generator,
                num_workers=2,
                pin_memory=torch.cuda.is_available(),
            )

            grad_accum = max(1, 16 // batch_size)
            updates_per_epoch = math.ceil(
                len(loader) / grad_accum
            )
            total_updates = (
                updates_per_epoch * E2E_EPOCHS
            )
            warmup_steps = int(round(
                total_updates * E2E_WARMUP_RATIO
            ))

            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=E2E_LR,
                weight_decay=E2E_WEIGHT_DECAY,
            )

            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_updates,
            )

            use_amp = torch.cuda.is_available()
            scaler = (
                torch.amp.GradScaler("cuda")
                if use_amp else None
            )

            history = []
            started = time.perf_counter()

            for epoch in range(E2E_EPOCHS):
                model.train()
                optimizer.zero_grad(set_to_none=True)
                losses = []

                for step, (ids, mask, labels) in enumerate(loader):
                    ids = ids.to(
                        DEVICE, non_blocking=True
                    )
                    mask = mask.to(
                        DEVICE, non_blocking=True
                    )
                    labels = labels.to(
                        DEVICE, non_blocking=True
                    )

                    with torch.amp.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                        enabled=use_amp,
                    ):
                        out = model(
                            input_ids=ids,
                            attention_mask=mask,
                            labels=labels,
                        )
                        loss = out.loss / grad_accum

                    if scaler is not None:
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()

                    losses.append(
                        float(loss.detach().cpu())
                        * grad_accum
                    )

                    do_step = (
                        (step + 1) % grad_accum == 0
                        or step + 1 == len(loader)
                    )

                    if do_step:
                        if scaler is not None:
                            scaler.unscale_(optimizer)

                        torch.nn.utils.clip_grad_norm_(
                            model.parameters(), 1.0
                        )

                        if scaler is not None:
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            optimizer.step()

                        scheduler.step()
                        optimizer.zero_grad(
                            set_to_none=True
                        )

                row = {
                    "model": model_name,
                    "seed": seed,
                    "label_budget": frac,
                    "epoch": epoch + 1,
                    "train_loss": float(
                        np.mean(losses)
                    ),
                    "batch_size": batch_size,
                    "gradient_accumulation": grad_accum,
                }
                history.append(row)
                print(row)

            fit_seconds = (
                time.perf_counter() - started
            )

            cal = score_e2e_model(
                model, TOKENS["calibration"]
            )
            iid = score_e2e_model(
                model, TOKENS["iid"]
            )
            holdout = score_e2e_model(
                model, TOKENS["holdout"]
            )

            np.savez_compressed(
                score_file,
                cal=cal,
                iid=iid,
                holdout=holdout,
                fit_seconds=np.asarray([
                    fit_seconds
                ]),
                batch_size=np.asarray([
                    batch_size
                ]),
                grad_accum=np.asarray([
                    grad_accum
                ]),
            )

            pd.DataFrame(history).to_csv(
                FINAL_AUDIT_ROOT
                / (
                    f"{model_name}_seed{seed}_"
                    f"budget{budget_tag(frac)}.csv"
                ),
                index=False,
            )

            del model
            del optimizer
            del scheduler
            del loader
            del ds
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            return

        except RuntimeError as exc:
            last_error = exc

            if "out of memory" not in str(exc).lower():
                raise

            print({
                "E2E_OOM": model_name,
                "seed": seed,
                "budget": frac,
                "failed_batch_size": batch_size,
            })

            try:
                del model
            except Exception:
                pass

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    raise RuntimeError(
        f"{model_name} Seed {seed} Budget {frac}: "
        "alle Batchgrößen fehlgeschlagen."
    ) from last_error


for seed in SEEDS:
    for frac in LABEL_BUDGETS:
        train_final_e2e(
            BASE_MODEL_DIR,
            seed,
            frac,
            "T0_E2E",
        )

        train_final_e2e(
            DAPT_ENCODERS[seed],
            seed,
            frac,
            "DAPT_E2E",
        )

In [ ]:
# ============================================================
# FINAL 05 – Gemeinsamer Score-Zugriff für alle 12 Systeme
# ============================================================

def load_score_npz(path):
    dat = np.load(path)
    return {
        "cal": np.asarray(dat["cal"]),
        "iid": np.asarray(dat["iid"]),
        "holdout": np.asarray(dat["holdout"]),
        "fit_seconds": (
            float(dat["fit_seconds"][0])
            if "fit_seconds" in dat.files
            else np.nan
        ),
    }

def final_model_scores(model, seed, frac):
    if model in FINAL_CORE_SYSTEMS:
        path = final_system_score_path(
            model, seed, frac
        )
        if not path.exists():
            raise FileNotFoundError(path)
        return load_score_npz(path)

    rep = None
    clf = None

    for r in REPRESENTATIONS:
        prefix = r + "_"
        if model.startswith(prefix):
            rep = r
            clf = model[len(prefix):]
            break

    if rep is None or clf not in CLASSIFIERS:
        raise KeyError(model)

    path = freeze_score_path(
        rep, clf, seed, frac
    )
    if not path.exists():
        raise FileNotFoundError(path)

    return load_score_npz(path)

missing = []

for model in FINAL_ALL_SYSTEMS:
    for seed in SEEDS:
        for frac in LABEL_BUDGETS:
            try:
                final_model_scores(
                    model, seed, frac
                )
            except Exception as exc:
                missing.append(
                    (model, seed, frac, str(exc))
                )

if missing:
    raise RuntimeError(
        f"FINAL Score-Coverage unvollständig: "
        f"{missing[:10]}"
    )

print({
    "systems": len(FINAL_ALL_SYSTEMS),
    "seeds": len(SEEDS),
    "budgets": len(LABEL_BUDGETS),
    "score_combinations": (
        len(FINAL_ALL_SYSTEMS)
        * len(SEEDS)
        * len(LABEL_BUDGETS)
    ),
    "coverage": "COMPLETE",
})

In [ ]:
# ============================================================
# FINAL 06 – MASTER RESULTS: FPR überall
# ============================================================

final_master_rows = []
final_ap_rows = []
final_ap_equal_n_rows = []

ycal = calibration_df[
    "label"
].to_numpy(dtype=int)

for model in FINAL_ALL_SYSTEMS:
    for seed in SEEDS:
        for frac in LABEL_BUDGETS:
            scores = final_model_scores(
                model, seed, frac
            )

            ops = calibration_operating_points(
                ycal, scores["cal"]
            )

            # AP schwellenwertfrei
            for scenario in [
                "IID_BALANCED",
                "TEMPORAL",
                "DOMAIN_OOD",
                "TEMPLATE_OOD",
                "DOMAIN_TEMPLATE_OOD",
            ]:
                y, s = scenario_score_view(
                    scores["iid"],
                    scores["holdout"],
                    scenario,
                )

                final_ap_rows.append({
                    "model": model,
                    "seed": seed,
                    "label_budget": frac,
                    "scenario": scenario,
                    "n_test": len(y),
                    "prevalence_phish": float(
                        (y == 1).mean()
                    ),
                    "average_precision": float(
                        average_precision_score(
                            y, s
                        )
                    ),
                    "roc_auc": float(
                        roc_auc_score(y, s)
                    ),
                })

                idx_eq = AP_EQUAL_N_INDICES[
                    scenario
                ]

                if scenario == "IID_BALANCED":
                    y_eq = iid_df.iloc[
                        idx_eq
                    ]["label"].to_numpy(
                        dtype=int
                    )
                    s_eq = scores["iid"][
                        idx_eq
                    ]
                else:
                    y_eq = holdout_df.iloc[
                        idx_eq
                    ]["label"].to_numpy(
                        dtype=int
                    )
                    s_eq = scores["holdout"][
                        idx_eq
                    ]

                final_ap_equal_n_rows.append({
                    "model": model,
                    "seed": seed,
                    "label_budget": frac,
                    "scenario": scenario,
                    "n_test": len(y_eq),
                    "prevalence_phish": float(
                        (y_eq == 1).mean()
                    ),
                    "average_precision": float(
                        average_precision_score(
                            y_eq, s_eq
                        )
                    ),
                    "roc_auc": float(
                        roc_auc_score(
                            y_eq, s_eq
                        )
                    ),
                })

            # FPR-/Threshold-Metriken
            for target in TARGET_FPRS:
                op = ops[target]

                for scenario in SCENARIOS:
                    y, s = scenario_score_view(
                        scores["iid"],
                        scores["holdout"],
                        scenario,
                    )

                    final_master_rows.append({
                        "model": model,
                        "seed": seed,
                        "label_budget": frac,
                        "n_train": len(
                            BUDGET_INDICES[
                                seed
                            ][frac]
                        ),
                        "scenario": scenario,
                        "n_test": len(y),
                        "target_fpr": target,
                        "threshold": op[
                            "threshold"
                        ],
                        "calibration_empirical_fpr": op[
                            "calibration_empirical_fpr"
                        ],
                        "calibration_recall": op[
                            "calibration_recall"
                        ],
                        "fit_seconds": scores[
                            "fit_seconds"
                        ],
                        **metric_row(
                            y,
                            s,
                            op["threshold"],
                        ),
                    })

FINAL_MASTER = pd.DataFrame(
    final_master_rows
)
FINAL_AP = pd.DataFrame(
    final_ap_rows
)
FINAL_AP_EQUAL_N = pd.DataFrame(
    final_ap_equal_n_rows
)

FINAL_MASTER.to_csv(
    FINAL_ROOT
    / "MASTER_threshold_results.csv",
    index=False,
)
FINAL_AP.to_csv(
    FINAL_ROOT
    / "MASTER_ap_prevalence_matched.csv",
    index=False,
)
FINAL_AP_EQUAL_N.to_csv(
    FINAL_ROOT
    / "MASTER_ap_equal_n.csv",
    index=False,
)

expected_master = (
    len(FINAL_ALL_SYSTEMS)
    * len(SEEDS)
    * len(LABEL_BUDGETS)
    * len(SCENARIOS)
    * len(TARGET_FPRS)
)

expected_ap = (
    len(FINAL_ALL_SYSTEMS)
    * len(SEEDS)
    * len(LABEL_BUDGETS)
    * 5
)

print({
    "master_rows": len(FINAL_MASTER),
    "expected_master": expected_master,
    "ap_rows": len(FINAL_AP),
    "expected_ap": expected_ap,
})

if (
    len(FINAL_MASTER) != expected_master
    or len(FINAL_AP) != expected_ap
):
    raise RuntimeError(
        "FINAL MASTER unvollständig."
    )

In [ ]:
# ============================================================
# FINAL 07 – Zusammenfassungen + LR/XGB/MLP-Rolle
# ============================================================

FINAL_SUMMARY = (
    FINAL_MASTER
    .groupby(
        [
            "model",
            "label_budget",
            "scenario",
            "target_fpr",
        ],
        as_index=False,
    )
    .agg(
        n_test=("n_test", "first"),
        prevalence_phish=(
            "prevalence_phish", "first"
        ),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        precision_mean=(
            "precision", "mean"
        ),
        f1_mean=("f1", "mean"),
        roc_auc_mean=("roc_auc", "mean"),
        empirical_fpr_mean=(
            "empirical_fpr", "mean"
        ),
        empirical_fpr_std=(
            "empirical_fpr", "std"
        ),
        fp_mean=("fp", "mean"),
        fn_mean=("fn", "mean"),
        fit_seconds_mean=(
            "fit_seconds", "mean"
        ),
    )
)

FINAL_AP_SUMMARY = (
    FINAL_AP
    .groupby(
        [
            "model",
            "label_budget",
            "scenario",
        ],
        as_index=False,
    )
    .agg(
        n_test=("n_test", "first"),
        average_precision_mean=(
            "average_precision", "mean"
        ),
        average_precision_std=(
            "average_precision", "std"
        ),
        roc_auc_mean=("roc_auc", "mean"),
    )
)

FINAL_SUMMARY.to_csv(
    FINAL_ROOT
    / "MASTER_threshold_summary.csv",
    index=False,
)
FINAL_AP_SUMMARY.to_csv(
    FINAL_ROOT
    / "MASTER_ap_summary.csv",
    index=False,
)

rep_models = [
    f"{rep}_{clf}"
    for rep in REPRESENTATIONS
    for clf in CLASSIFIERS
]

clf_data = FINAL_MASTER[
    FINAL_MASTER["model"].isin(
        rep_models
    )
    & FINAL_MASTER["scenario"].isin(
        FINAL_STRESS
    )
    & FINAL_MASTER[
        "target_fpr"
    ].eq(PRIMARY_TARGET_FPR)
].copy()

clf_data[
    ["representation", "classifier"]
] = clf_data["model"].str.extract(
    r"^(BASE|DAPT|CONTRASTIVE)_"
    r"(LOGREG|XGBOOST|MLP)$"
)

CLASSIFIER_ROLE_SUMMARY = (
    clf_data
    .groupby(
        [
            "representation",
            "classifier",
            "label_budget",
        ],
        as_index=False,
    )
    .agg(
        mean_stress_recall=(
            "recall", "mean"
        ),
        mean_stress_fpr=(
            "empirical_fpr", "mean"
        ),
        recall_variability=(
            "recall", "std"
        ),
    )
)

CLASSIFIER_ROLE_SUMMARY.to_csv(
    FINAL_ROOT
    / "classifier_family_role_summary.csv",
    index=False,
)

display(CLASSIFIER_ROLE_SUMMARY)

In [ ]:
# ============================================================
# FINAL 08 – Korrigierte AP-Shift-Analyse
# ============================================================

def ap_shift_table(df):
    rows = []

    for keys, g in df.groupby(
        ["model", "seed", "label_budget"]
    ):
        iid = g[
            g["scenario"]
            == "IID_BALANCED"
        ]

        if len(iid) != 1:
            raise RuntimeError(
                f"IID_BALANCED fehlt: {keys}"
            )

        iid = iid.iloc[0]

        for scenario in FINAL_STRESS:
            ood = g[
                g["scenario"] == scenario
            ]

            if len(ood) != 1:
                raise RuntimeError(
                    f"{scenario} fehlt: {keys}"
                )

            ood = ood.iloc[0]

            rows.append({
                "model": keys[0],
                "seed": keys[1],
                "label_budget": keys[2],
                "comparison": (
                    "IID_BALANCED->"
                    + scenario
                ),
                "iid_n": int(iid["n_test"]),
                "ood_n": int(ood["n_test"]),
                "iid_ap": float(
                    iid["average_precision"]
                ),
                "ood_ap": float(
                    ood["average_precision"]
                ),
                "ap_delta_ood_minus_iid": float(
                    ood["average_precision"]
                    - iid["average_precision"]
                ),
            })

    return pd.DataFrame(rows)

FINAL_AP_SHIFT = ap_shift_table(
    FINAL_AP
)
FINAL_AP_SHIFT_EQUAL_N = ap_shift_table(
    FINAL_AP_EQUAL_N
)

FINAL_AP_SHIFT.to_csv(
    FINAL_ROOT
    / "AP_SHIFT_prevalence_matched.csv",
    index=False,
)
FINAL_AP_SHIFT_EQUAL_N.to_csv(
    FINAL_ROOT
    / "AP_SHIFT_equal_n.csv",
    index=False,
)

print({
    "ap_shift_rows": len(
        FINAL_AP_SHIFT
    ),
    "equal_n_rows": len(
        FINAL_AP_SHIFT_EQUAL_N
    ),
})

In [ ]:
# ============================================================
# FINAL 09 – Statistik über ALLE fünf Seeds
# ============================================================

FINAL_CONTRASTS = [
    (
        "DAPT_E2E",
        "T0_E2E",
        "DAPT_E2E_minus_T0_E2E",
    ),
    (
        "DAPT_E2E",
        "B0_STRUCT_XGB",
        "DAPT_E2E_minus_B0",
    ),
    (
        "T0_E2E",
        "B0_STRUCT_XGB",
        "T0_E2E_minus_B0",
    ),
    (
        "DAPT_LOGREG",
        "BASE_LOGREG",
        "DAPT_minus_BASE_LOGREG",
    ),
    (
        "DAPT_XGBOOST",
        "BASE_XGBOOST",
        "DAPT_minus_BASE_XGBOOST",
    ),
    (
        "DAPT_MLP",
        "BASE_MLP",
        "DAPT_minus_BASE_MLP",
    ),
    (
        "CONTRASTIVE_LOGREG",
        "BASE_LOGREG",
        "CONTRASTIVE_minus_BASE_LOGREG",
    ),
    (
        "CONTRASTIVE_XGBOOST",
        "BASE_XGBOOST",
        "CONTRASTIVE_minus_BASE_XGBOOST",
    ),
    (
        "CONTRASTIVE_MLP",
        "BASE_MLP",
        "CONTRASTIVE_minus_BASE_MLP",
    ),
]

def exact_signflip_p(diff):
    d = np.asarray(
        diff, dtype=float
    )
    d = d[np.isfinite(d)]

    if len(d) == 0:
        return np.nan

    observed = abs(d.mean())

    permuted = []

    for signs in product(
        [-1.0, 1.0],
        repeat=len(d),
    ):
        permuted.append(
            abs(
                np.mean(
                    d
                    * np.asarray(signs)
                )
            )
        )

    permuted = np.asarray(permuted)

    return float(
        np.mean(
            permuted
            >= observed - 1e-15
        )
    )

def paired_seed_stats(
    diff,
    higher_is_better=True,
):
    d = np.asarray(
        diff, dtype=float
    )
    d = d[np.isfinite(d)]

    n = len(d)

    if n == 0:
        return {}

    mean = float(d.mean())

    sd = (
        float(d.std(ddof=1))
        if n > 1
        else np.nan
    )

    sem = (
        sd / math.sqrt(n)
        if n > 1
        else np.nan
    )

    crit = (
        float(
            student_t.ppf(
                0.975,
                df=n - 1,
            )
        )
        if n > 1
        else np.nan
    )

    if higher_is_better:
        wins = int(
            (d > 1e-12).sum()
        )
        losses = int(
            (d < -1e-12).sum()
        )
    else:
        wins = int(
            (d < -1e-12).sum()
        )
        losses = int(
            (d > 1e-12).sum()
        )

    ties = n - wins - losses

    return {
        "n_seeds": n,
        "mean_difference": mean,
        "sd_difference": sd,
        "ci95_low": (
            mean - crit * sem
            if n > 1
            else np.nan
        ),
        "ci95_high": (
            mean + crit * sem
            if n > 1
            else np.nan
        ),
        "exact_signflip_p_two_sided": (
            exact_signflip_p(d)
        ),
        "cohen_dz": (
            float(mean / sd)
            if n > 1 and sd > 0
            else np.nan
        ),
        "wins": wins,
        "ties": ties,
        "losses": losses,
        "same_direction_5_of_5": bool(
            wins == n or losses == n
        ),
    }

final_stat_rows = []

for candidate, reference, contrast in (
    FINAL_CONTRASTS
):
    for frac in LABEL_BUDGETS:
        for scenario in SCENARIOS:
            for target in TARGET_FPRS:
                c = FINAL_MASTER[
                    (FINAL_MASTER.model == candidate)
                    & (
                        FINAL_MASTER.label_budget
                        == frac
                    )
                    & (
                        FINAL_MASTER.scenario
                        == scenario
                    )
                    & (
                        FINAL_MASTER.target_fpr
                        == target
                    )
                ].set_index("seed")

                r = FINAL_MASTER[
                    (FINAL_MASTER.model == reference)
                    & (
                        FINAL_MASTER.label_budget
                        == frac
                    )
                    & (
                        FINAL_MASTER.scenario
                        == scenario
                    )
                    & (
                        FINAL_MASTER.target_fpr
                        == target
                    )
                ].set_index("seed")

                common = sorted(
                    set(c.index)
                    & set(r.index)
                )

                if common != SEEDS:
                    raise RuntimeError(
                        "Statistik-Seed-Coverage "
                        f"unvollständig: {contrast}"
                    )

                for metric, higher in [
                    ("recall", True),
                    (
                        "empirical_fpr",
                        False,
                    ),
                    ("f1", True),
                    ("precision", True),
                    ("roc_auc", True),
                ]:
                    diff = (
                        c.loc[
                            common, metric
                        ].to_numpy()
                        - r.loc[
                            common, metric
                        ].to_numpy()
                    )

                    final_stat_rows.append({
                        "contrast": contrast,
                        "candidate": candidate,
                        "reference": reference,
                        "label_budget": frac,
                        "scenario": scenario,
                        "target_fpr": target,
                        "metric": metric,
                        "higher_is_better": higher,
                        **paired_seed_stats(
                            diff,
                            higher_is_better=higher,
                        ),
                    })

# AP separat
for candidate, reference, contrast in (
    FINAL_CONTRASTS
):
    for frac in LABEL_BUDGETS:
        for scenario in [
            "IID_BALANCED",
            *FINAL_STRESS,
        ]:
            c = FINAL_AP[
                (FINAL_AP.model == candidate)
                & (
                    FINAL_AP.label_budget
                    == frac
                )
                & (
                    FINAL_AP.scenario
                    == scenario
                )
            ].set_index("seed")

            r = FINAL_AP[
                (FINAL_AP.model == reference)
                & (
                    FINAL_AP.label_budget
                    == frac
                )
                & (
                    FINAL_AP.scenario
                    == scenario
                )
            ].set_index("seed")

            common = sorted(
                set(c.index)
                & set(r.index)
            )

            diff = (
                c.loc[
                    common,
                    "average_precision",
                ].to_numpy()
                - r.loc[
                    common,
                    "average_precision",
                ].to_numpy()
            )

            final_stat_rows.append({
                "contrast": contrast,
                "candidate": candidate,
                "reference": reference,
                "label_budget": frac,
                "scenario": scenario,
                "target_fpr": np.nan,
                "metric": "average_precision",
                "higher_is_better": True,
                **paired_seed_stats(
                    diff,
                    higher_is_better=True,
                ),
            })

FINAL_STATS = pd.DataFrame(
    final_stat_rows
)

FINAL_STATS.to_csv(
    FINAL_ROOT
    / "STATISTICS_paired_seed_tests.csv",
    index=False,
)

print({
    "statistical_rows": len(
        FINAL_STATS
    ),
    "seeds_used": SEEDS,
    "seeds_excluded": [],
    "minimum_possible_exact_two_sided_p": (
        2 / (2 ** len(SEEDS))
    ),
})

In [ ]:
# ============================================================
# FINAL 10 – Modellranking und Stabilität
# ============================================================

rank_rows = []

ranking_source = FINAL_MASTER[
    FINAL_MASTER[
        "scenario"
    ].isin(FINAL_STRESS)
    & FINAL_MASTER[
        "target_fpr"
    ].eq(PRIMARY_TARGET_FPR)
].copy()

for (
    frac,
    scenario,
    seed,
), part in ranking_source.groupby(
    [
        "label_budget",
        "scenario",
        "seed",
    ]
):
    p = part.copy()

    p["recall_rank"] = p[
        "recall"
    ].rank(
        method="average",
        ascending=False,
    )

    for _, row in p.iterrows():
        rank_rows.append({
            "model": row["model"],
            "seed": seed,
            "label_budget": frac,
            "scenario": scenario,
            "recall": row["recall"],
            "empirical_fpr": row[
                "empirical_fpr"
            ],
            "recall_rank": row[
                "recall_rank"
            ],
        })

FINAL_RANK = pd.DataFrame(
    rank_rows
)

FINAL_RANK_SUMMARY = (
    FINAL_RANK
    .groupby(
        ["model", "label_budget"],
        as_index=False,
    )
    .agg(
        mean_rank=(
            "recall_rank", "mean"
        ),
        median_rank=(
            "recall_rank", "median"
        ),
        mean_stress_recall=(
            "recall", "mean"
        ),
        worst_stress_recall=(
            "recall", "min"
        ),
        mean_stress_fpr=(
            "empirical_fpr", "mean"
        ),
        worst_stress_fpr=(
            "empirical_fpr", "max"
        ),
    )
)

FINAL_RANK.to_csv(
    FINAL_ROOT
    / "MODEL_rank_by_seed_scenario.csv",
    index=False,
)

FINAL_RANK_SUMMARY.to_csv(
    FINAL_ROOT
    / "MODEL_rank_stability_summary.csv",
    index=False,
)

display(
    FINAL_RANK_SUMMARY.sort_values(
        ["label_budget", "mean_rank"]
    ).head(40)
)

In [ ]:
# ============================================================
# FINAL 11 – Kaskade für 10 / 25 / 100 %
# ============================================================

FINAL_CASCADE_RANKERS = (
    ["B0_SELF"]
    + [
        m
        for m in FINAL_ALL_SYSTEMS
        if m != "B0_STRUCT_XGB"
    ]
)

final_cascade_rows = []

for seed in SEEDS:
    for frac in LABEL_BUDGETS:
        b0 = final_model_scores(
            "B0_STRUCT_XGB",
            seed,
            frac,
        )

        b0_ops = calibration_operating_points(
            ycal,
            b0["cal"],
        )

        ranker_cache = {}
        ranker_ops = {}

        for ranker in FINAL_CASCADE_RANKERS:
            source_model = (
                "B0_STRUCT_XGB"
                if ranker == "B0_SELF"
                else ranker
            )

            rs = final_model_scores(
                source_model,
                seed,
                frac,
            )

            ranker_cache[ranker] = rs
            ranker_ops[ranker] = (
                calibration_operating_points(
                    ycal,
                    rs["cal"],
                )
            )

        for target in TARGET_FPRS:
            b0_op = b0_ops[target]

            for scenario in SCENARIOS:
                y, b0_score = (
                    scenario_score_view(
                        b0["iid"],
                        b0["holdout"],
                        scenario,
                    )
                )

                b0_pred = (
                    b0_score
                    >= b0_op["threshold"]
                )

                stage1_negative = ~b0_pred

                neg_idx = np.flatnonzero(
                    stage1_negative
                )

                tn, fp, fn, tp = (
                    confusion_matrix(
                        y,
                        b0_pred.astype(int),
                        labels=[0, 1],
                    ).ravel()
                )

                b0_fpr = (
                    fp / max(fp + tn, 1)
                )

                b0_recall = (
                    tp / max(tp + fn, 1)
                )

                base_fn_mask = (
                    (y == 1)
                    & stage1_negative
                )

                n_base_fn = int(
                    base_fn_mask.sum()
                )

                for review_frac in (
                    FINAL_REVIEW_FRACTIONS
                ):
                    n_review = max(
                        1,
                        int(
                            math.ceil(
                                review_frac
                                * len(neg_idx)
                            )
                        ),
                    )

                    for ranker in (
                        FINAL_CASCADE_RANKERS
                    ):
                        rs = ranker_cache[
                            ranker
                        ]

                        _, rank_score = (
                            scenario_score_view(
                                rs["iid"],
                                rs["holdout"],
                                scenario,
                            )
                        )

                        ordered = neg_idx[
                            np.argsort(
                                rank_score[
                                    neg_idx
                                ]
                            )[::-1]
                        ]

                        review_idx = (
                            ordered[:n_review]
                        )

                        rescued = int(
                            base_fn_mask[
                                review_idx
                            ].sum()
                        )

                        review_phish = int(
                            (
                                y[review_idx]
                                == 1
                            ).sum()
                        )

                        rop = ranker_ops[
                            ranker
                        ][target]

                        rank_pred = (
                            rank_score
                            >= rop["threshold"]
                        )

                        r_tn, r_fp, _, _ = (
                            confusion_matrix(
                                y,
                                rank_pred.astype(
                                    int
                                ),
                                labels=[0, 1],
                            ).ravel()
                        )

                        rank_fpr = (
                            r_fp
                            / max(
                                r_fp + r_tn,
                                1,
                            )
                        )

                        assisted_recall = (
                            (tp + rescued)
                            / max(tp + fn, 1)
                        )

                        final_cascade_rows.append({
                            "seed": seed,
                            "label_budget": frac,
                            "scenario": scenario,
                            "target_fpr": target,
                            "stage1_model": (
                                "B0_STRUCT_XGB"
                            ),
                            "stage1_empirical_fpr": (
                                b0_fpr
                            ),
                            "stage1_recall": (
                                b0_recall
                            ),
                            "stage1_fp": int(fp),
                            "stage1_fn": int(fn),
                            "ranker": ranker,
                            "ranker_empirical_fpr_at_own_threshold": (
                                rank_fpr
                            ),
                            "review_fraction_of_stage1_negatives": (
                                review_frac
                            ),
                            "review_n": (
                                n_review
                            ),
                            "review_fraction_of_all_cases": (
                                n_review
                                / len(y)
                            ),
                            "review_precision": (
                                review_phish
                                / max(
                                    n_review,
                                    1,
                                )
                            ),
                            "rescued_stage1_fn": (
                                rescued
                            ),
                            "capture_rate_of_stage1_fn": (
                                rescued
                                / max(
                                    n_base_fn,
                                    1,
                                )
                            ),
                            "review_assisted_recall_upper_bound": (
                                assisted_recall
                            ),
                            "potential_recall_gain_pp": (
                                100
                                * (
                                    assisted_recall
                                    - b0_recall
                                )
                            ),
                        })

FINAL_CASCADE = pd.DataFrame(
    final_cascade_rows
)

self_ref = (
    FINAL_CASCADE[
        FINAL_CASCADE[
            "ranker"
        ] == "B0_SELF"
    ][
        [
            "seed",
            "label_budget",
            "scenario",
            "target_fpr",
            "review_fraction_of_stage1_negatives",
            "rescued_stage1_fn",
            "review_precision",
            "review_assisted_recall_upper_bound",
        ]
    ]
    .rename(columns={
        "rescued_stage1_fn": (
            "self_rescued_stage1_fn"
        ),
        "review_precision": (
            "self_review_precision"
        ),
        "review_assisted_recall_upper_bound": (
            "self_assisted_recall"
        ),
    })
)

FINAL_CASCADE = FINAL_CASCADE.merge(
    self_ref,
    on=[
        "seed",
        "label_budget",
        "scenario",
        "target_fpr",
        "review_fraction_of_stage1_negatives",
    ],
    how="left",
)

FINAL_CASCADE[
    "rescued_fn_lift_vs_self"
] = (
    FINAL_CASCADE[
        "rescued_stage1_fn"
    ]
    - FINAL_CASCADE[
        "self_rescued_stage1_fn"
    ]
)

FINAL_CASCADE[
    "assisted_recall_lift_vs_self_pp"
] = 100 * (
    FINAL_CASCADE[
        "review_assisted_recall_upper_bound"
    ]
    - FINAL_CASCADE[
        "self_assisted_recall"
    ]
)

FINAL_CASCADE.to_csv(
    FINAL_ROOT
    / "CASCADE_results.csv",
    index=False,
)

FINAL_CASCADE_SUMMARY = (
    FINAL_CASCADE
    .groupby(
        [
            "label_budget",
            "scenario",
            "target_fpr",
            "ranker",
            "review_fraction_of_stage1_negatives",
        ],
        as_index=False,
    )
    .agg(
        stage1_empirical_fpr_mean=(
            "stage1_empirical_fpr",
            "mean",
        ),
        stage1_recall_mean=(
            "stage1_recall",
            "mean",
        ),
        ranker_empirical_fpr_mean=(
            "ranker_empirical_fpr_at_own_threshold",
            "mean",
        ),
        review_precision_mean=(
            "review_precision",
            "mean",
        ),
        rescued_stage1_fn_mean=(
            "rescued_stage1_fn",
            "mean",
        ),
        capture_rate_mean=(
            "capture_rate_of_stage1_fn",
            "mean",
        ),
        assisted_recall_mean=(
            "review_assisted_recall_upper_bound",
            "mean",
        ),
        potential_recall_gain_pp_mean=(
            "potential_recall_gain_pp",
            "mean",
        ),
        rescued_fn_lift_vs_self_mean=(
            "rescued_fn_lift_vs_self",
            "mean",
        ),
        assisted_recall_lift_vs_self_pp_mean=(
            "assisted_recall_lift_vs_self_pp",
            "mean",
        ),
    )
)

FINAL_CASCADE_SUMMARY.to_csv(
    FINAL_ROOT
    / "CASCADE_summary.csv",
    index=False,
)

print({
    "cascade_rows": len(
        FINAL_CASCADE
    )
})

In [ ]:
# ============================================================
# FINAL 12 – Fehlerkomplementarität für alle Budgets
# ============================================================

final_error_rows = []

for seed in SEEDS:
    for frac in LABEL_BUDGETS:
        b0 = final_model_scores(
            "B0_STRUCT_XGB",
            seed,
            frac,
        )

        b0_ops = (
            calibration_operating_points(
                ycal,
                b0["cal"],
            )
        )

        for candidate in [
            m
            for m in FINAL_ALL_SYSTEMS
            if m != "B0_STRUCT_XGB"
        ]:
            cand = final_model_scores(
                candidate,
                seed,
                frac,
            )

            cand_ops = (
                calibration_operating_points(
                    ycal,
                    cand["cal"],
                )
            )

            for target in TARGET_FPRS:
                for scenario in SCENARIOS:
                    y, bs = (
                        scenario_score_view(
                            b0["iid"],
                            b0["holdout"],
                            scenario,
                        )
                    )

                    _, cs = (
                        scenario_score_view(
                            cand["iid"],
                            cand["holdout"],
                            scenario,
                        )
                    )

                    bp = (
                        bs
                        >= b0_ops[
                            target
                        ]["threshold"]
                    )

                    cp = (
                        cs
                        >= cand_ops[
                            target
                        ]["threshold"]
                    )

                    b_fn = (
                        (y == 1)
                        & (~bp)
                    )
                    c_fn = (
                        (y == 1)
                        & (~cp)
                    )
                    b_fp = (
                        (y == 0)
                        & bp
                    )
                    c_fp = (
                        (y == 0)
                        & cp
                    )

                    common_fn = int(
                        (b_fn & c_fn).sum()
                    )

                    rescued = int(
                        (
                            b_fn
                            & (~c_fn)
                        ).sum()
                    )

                    regressed = int(
                        (
                            (y == 1)
                            & bp
                            & c_fn
                        ).sum()
                    )

                    common_fp = int(
                        (b_fp & c_fp).sum()
                    )

                    avoided_fp = int(
                        (
                            b_fp
                            & (~c_fp)
                        ).sum()
                    )

                    new_fp = int(
                        (
                            (y == 0)
                            & (~bp)
                            & c_fp
                        ).sum()
                    )

                    final_error_rows.append({
                        "seed": seed,
                        "label_budget": frac,
                        "scenario": scenario,
                        "target_fpr": target,
                        "base_model": (
                            "B0_STRUCT_XGB"
                        ),
                        "candidate_model": (
                            candidate
                        ),
                        "base_empirical_fpr": (
                            b_fp.sum()
                            / max(
                                (y == 0).sum(),
                                1,
                            )
                        ),
                        "candidate_empirical_fpr": (
                            c_fp.sum()
                            / max(
                                (y == 0).sum(),
                                1,
                            )
                        ),
                        "base_fn": int(
                            b_fn.sum()
                        ),
                        "candidate_fn": int(
                            c_fn.sum()
                        ),
                        "common_fn": (
                            common_fn
                        ),
                        "rescued_b0_fn": (
                            rescued
                        ),
                        "regressed_b0_tp": (
                            regressed
                        ),
                        "fn_rescue_rate": (
                            rescued
                            / max(
                                b_fn.sum(),
                                1,
                            )
                        ),
                        "fn_jaccard": (
                            common_fn
                            / max(
                                (
                                    b_fn
                                    | c_fn
                                ).sum(),
                                1,
                            )
                        ),
                        "base_fp": int(
                            b_fp.sum()
                        ),
                        "candidate_fp": int(
                            c_fp.sum()
                        ),
                        "common_fp": (
                            common_fp
                        ),
                        "avoided_b0_fp": (
                            avoided_fp
                        ),
                        "new_fp_vs_b0": (
                            new_fp
                        ),
                        "fp_jaccard": (
                            common_fp
                            / max(
                                (
                                    b_fp
                                    | c_fp
                                ).sum(),
                                1,
                            )
                        ),
                    })

FINAL_ERROR = pd.DataFrame(
    final_error_rows
)

FINAL_ERROR.to_csv(
    FINAL_ROOT
    / "ERROR_complementarity.csv",
    index=False,
)

FINAL_ERROR_SUMMARY = (
    FINAL_ERROR
    .groupby(
        [
            "label_budget",
            "scenario",
            "target_fpr",
            "candidate_model",
        ],
        as_index=False,
    )
    .agg(
        base_empirical_fpr_mean=(
            "base_empirical_fpr",
            "mean",
        ),
        candidate_empirical_fpr_mean=(
            "candidate_empirical_fpr",
            "mean",
        ),
        base_fn_mean=(
            "base_fn", "mean"
        ),
        candidate_fn_mean=(
            "candidate_fn", "mean"
        ),
        rescued_b0_fn_mean=(
            "rescued_b0_fn",
            "mean",
        ),
        regressed_b0_tp_mean=(
            "regressed_b0_tp",
            "mean",
        ),
        fn_rescue_rate_mean=(
            "fn_rescue_rate",
            "mean",
        ),
        fn_jaccard_mean=(
            "fn_jaccard",
            "mean",
        ),
        base_fp_mean=(
            "base_fp", "mean"
        ),
        candidate_fp_mean=(
            "candidate_fp", "mean"
        ),
        avoided_b0_fp_mean=(
            "avoided_b0_fp",
            "mean",
        ),
        new_fp_vs_b0_mean=(
            "new_fp_vs_b0",
            "mean",
        ),
        fp_jaccard_mean=(
            "fp_jaccard",
            "mean",
        ),
    )
)

FINAL_ERROR_SUMMARY.to_csv(
    FINAL_ROOT
    / "ERROR_complementarity_summary.csv",
    index=False,
)

print({
    "error_rows": len(
        FINAL_ERROR
    )
})

## Explorative Active-Learning-Erweiterung

Active Learning verändert weder die Forschungsfrage noch den FINAL-FREEZE-Kern.

Es wird ausschließlich auf dem bestehenden **4.000-Fälle-Trainingspool** gearbeitet.
Der Startpunkt entspricht dem 10-%-Subset (400 Labels). Danach werden jeweils 200
weitere Fälle ausgewählt, bis 25 % beziehungsweise 1.000 Labels erreicht sind.

`UNCERTAINTY` wählt die Fälle mit dem geringsten Abstand der vorhergesagten
Phishing-Wahrscheinlichkeit zu 0,5. `RANDOM` verwendet pro Seed dieselbe zufällige
Kontrollreihenfolge für alle Repräsentationen und Klassifikatoren.

Die Labels der neuen Fälle werden erst nach ihrer Auswahl aus dem Trainingsdatensatz
offengelegt. Calibration, IID und OOD bleiben reine Evaluationsdaten.

In [ ]:
# ============================================================
# FINAL 13 – Active Learning: UNCERTAINTY vs RANDOM
# ============================================================

AL_RESULT_PATH = (
    FINAL_ROOT
    / "ACTIVE_LEARNING_results.csv"
)

AL_SELECTION_PATH = (
    FINAL_AUDIT_ROOT
    / "active_learning_selected_indices.json"
)

if AL_RESULT_PATH.exists():
    al_existing = pd.read_csv(
        AL_RESULT_PATH
    )
else:
    al_existing = pd.DataFrame()

if AL_SELECTION_PATH.exists():
    al_selection_store = json.loads(
        AL_SELECTION_PATH.read_text(
            encoding="utf-8"
        )
    )
else:
    al_selection_store = {}

def al_config_complete(
    rep,
    clf,
    seed,
    strategy,
):
    if al_existing.empty:
        return False

    p = al_existing[
        (
            al_existing[
                "representation"
            ] == rep
        )
        & (
            al_existing[
                "classifier"
            ] == clf
        )
        & (
            al_existing[
                "seed"
            ] == seed
        )
        & (
            al_existing[
                "strategy"
            ] == strategy
        )
    ]

    expected = (
        len(AL_BUDGETS)
        * len(SCENARIOS)
        * len(TARGET_FPRS)
    )

    return len(p) == expected

new_al_rows = []

for rep in REPRESENTATIONS:
    for clf in CLASSIFIERS:
        for seed in SEEDS:
            Xall = load_emb(
                rep, seed, "train"
            )
            Xcal = load_emb(
                rep, seed, "calibration"
            )
            Xiid = load_emb(
                rep, seed, "iid"
            )
            Xhold = load_emb(
                rep, seed, "holdout"
            )

            initial = (
                BUDGET_INDICES[
                    seed
                ][0.10].copy()
            )

            # Eine identische Random-Kontrollreihenfolge
            # pro Seed für alle Repräsentationen/Classifier.
            random_order = np.setdiff1d(
                np.arange(
                    len(y_train),
                    dtype=np.int32,
                ),
                initial,
            )

            rng = np.random.default_rng(
                seed + 9000
            )
            rng.shuffle(random_order)

            for strategy in AL_STRATEGIES:
                if al_config_complete(
                    rep,
                    clf,
                    seed,
                    strategy,
                ):
                    print({
                        "AL_REUSE": [
                            rep,
                            clf,
                            seed,
                            strategy,
                        ]
                    })
                    continue

                labeled = initial.copy()
                unlabeled = np.setdiff1d(
                    np.arange(
                        len(y_train),
                        dtype=np.int32,
                    ),
                    labeled,
                )

                random_cursor = 0
                selection_history = {
                    "0.10": labeled.tolist()
                }

                for round_i, frac in enumerate(
                    AL_BUDGETS
                ):
                    expected_n = int(
                        round(
                            len(y_train)
                            * frac
                        )
                    )

                    if len(labeled) != expected_n:
                        raise RuntimeError(
                            "AL Labelzahl falsch: "
                            f"{rep}/{clf}/{seed}/"
                            f"{strategy}/{frac}: "
                            f"{len(labeled)} "
                            f"statt {expected_n}"
                        )

                    model = build_classifier(
                        clf,
                        BEST_PARAMS[clf],
                        seed + round_i,
                    )

                    model.fit(
                        Xall[labeled],
                        y_train[labeled],
                    )

                    cal_score = (
                        model.predict_proba(
                            Xcal
                        )[:, 1]
                    )

                    iid_score = (
                        model.predict_proba(
                            Xiid
                        )[:, 1]
                    )

                    hold_score = (
                        model.predict_proba(
                            Xhold
                        )[:, 1]
                    )

                    ops = (
                        calibration_operating_points(
                            ycal,
                            cal_score,
                        )
                    )

                    for target in TARGET_FPRS:
                        op = ops[target]

                        for scenario in SCENARIOS:
                            y, s = (
                                scenario_score_view(
                                    iid_score,
                                    hold_score,
                                    scenario,
                                )
                            )

                            new_al_rows.append({
                                "representation": (
                                    rep
                                ),
                                "classifier": (
                                    clf
                                ),
                                "seed": seed,
                                "strategy": (
                                    strategy
                                ),
                                "label_budget": (
                                    frac
                                ),
                                "n_labeled": (
                                    len(labeled)
                                ),
                                "labeled_benign": int(
                                    (
                                        y_train[
                                            labeled
                                        ]
                                        == 0
                                    ).sum()
                                ),
                                "labeled_phish": int(
                                    (
                                        y_train[
                                            labeled
                                        ]
                                        == 1
                                    ).sum()
                                ),
                                "scenario": (
                                    scenario
                                ),
                                "target_fpr": (
                                    target
                                ),
                                "threshold": (
                                    op["threshold"]
                                ),
                                "calibration_empirical_fpr": (
                                    op[
                                        "calibration_empirical_fpr"
                                    ]
                                ),
                                **metric_row(
                                    y,
                                    s,
                                    op["threshold"],
                                ),
                            })

                    if frac == AL_BUDGETS[-1]:
                        del model
                        gc.collect()
                        break

                    if strategy == "UNCERTAINTY":
                        pool_score = (
                            model.predict_proba(
                                Xall[
                                    unlabeled
                                ]
                            )[:, 1]
                        )

                        uncertainty = np.abs(
                            pool_score - 0.5
                        )

                        chosen_local = (
                            np.argsort(
                                uncertainty
                            )[:AL_ACQUIRE_N]
                        )

                        chosen = unlabeled[
                            chosen_local
                        ]

                    elif strategy == "RANDOM":
                        # random_order enthält bereits nur
                        # Elemente außerhalb des Initialsets.
                        chosen = random_order[
                            random_cursor:
                            random_cursor
                            + AL_ACQUIRE_N
                        ]
                        random_cursor += (
                            AL_ACQUIRE_N
                        )

                    else:
                        raise KeyError(strategy)

                    # Sicherheitscheck:
                    if not set(
                        chosen.tolist()
                    ).issubset(
                        set(
                            unlabeled.tolist()
                        )
                    ):
                        raise RuntimeError(
                            "AL hat bereits "
                            "gelabelte Fälle gewählt."
                        )

                    labeled = np.sort(
                        np.concatenate(
                            [labeled, chosen]
                        )
                    ).astype(np.int32)

                    unlabeled = np.setdiff1d(
                        unlabeled,
                        chosen,
                    )

                    next_frac = AL_BUDGETS[
                        round_i + 1
                    ]

                    selection_history[
                        f"{next_frac:.2f}"
                    ] = labeled.tolist()

                    del model
                    gc.collect()

                al_selection_store[
                    (
                        f"{rep}|{clf}|"
                        f"{seed}|{strategy}"
                    )
                ] = selection_history

                current = pd.concat(
                    [
                        al_existing,
                        pd.DataFrame(
                            new_al_rows
                        ),
                    ],
                    ignore_index=True,
                )

                current.to_csv(
                    AL_RESULT_PATH,
                    index=False,
                )

                AL_SELECTION_PATH.write_text(
                    json.dumps(
                        al_selection_store
                    ),
                    encoding="utf-8",
                )

            del Xall
            del Xcal
            del Xiid
            del Xhold
            gc.collect()

ACTIVE_LEARNING = pd.read_csv(
    AL_RESULT_PATH
)

expected_al = (
    len(REPRESENTATIONS)
    * len(CLASSIFIERS)
    * len(SEEDS)
    * len(AL_STRATEGIES)
    * len(AL_BUDGETS)
    * len(SCENARIOS)
    * len(TARGET_FPRS)
)

print({
    "active_learning_rows": len(
        ACTIVE_LEARNING
    ),
    "expected": expected_al,
})

if len(ACTIVE_LEARNING) != expected_al:
    raise RuntimeError(
        "Active-Learning-Auswertung "
        "unvollständig."
    )

In [ ]:
# ============================================================
# FINAL 14 – Active-Learning-Zusammenfassung + Statistik
# ============================================================

AL_SUMMARY = (
    ACTIVE_LEARNING
    .groupby(
        [
            "representation",
            "classifier",
            "strategy",
            "label_budget",
            "scenario",
            "target_fpr",
        ],
        as_index=False,
    )
    .agg(
        recall_mean=(
            "recall", "mean"
        ),
        recall_std=(
            "recall", "std"
        ),
        empirical_fpr_mean=(
            "empirical_fpr", "mean"
        ),
        empirical_fpr_std=(
            "empirical_fpr", "std"
        ),
        average_precision_mean=(
            "average_precision", "mean"
        ),
        f1_mean=("f1", "mean"),
        labeled_phish_mean=(
            "labeled_phish", "mean"
        ),
        labeled_benign_mean=(
            "labeled_benign", "mean"
        ),
    )
)

AL_SUMMARY.to_csv(
    FINAL_ROOT
    / "ACTIVE_LEARNING_summary.csv",
    index=False,
)

al_stat_rows = []

for rep in REPRESENTATIONS:
    for clf in CLASSIFIERS:
        for frac in AL_BUDGETS:
            for scenario in SCENARIOS:
                for target in TARGET_FPRS:
                    u = ACTIVE_LEARNING[
                        (
                            ACTIVE_LEARNING[
                                "representation"
                            ] == rep
                        )
                        & (
                            ACTIVE_LEARNING[
                                "classifier"
                            ] == clf
                        )
                        & (
                            ACTIVE_LEARNING[
                                "strategy"
                            ] == "UNCERTAINTY"
                        )
                        & (
                            ACTIVE_LEARNING[
                                "label_budget"
                            ] == frac
                        )
                        & (
                            ACTIVE_LEARNING[
                                "scenario"
                            ] == scenario
                        )
                        & (
                            ACTIVE_LEARNING[
                                "target_fpr"
                            ] == target
                        )
                    ].set_index("seed")

                    r = ACTIVE_LEARNING[
                        (
                            ACTIVE_LEARNING[
                                "representation"
                            ] == rep
                        )
                        & (
                            ACTIVE_LEARNING[
                                "classifier"
                            ] == clf
                        )
                        & (
                            ACTIVE_LEARNING[
                                "strategy"
                            ] == "RANDOM"
                        )
                        & (
                            ACTIVE_LEARNING[
                                "label_budget"
                            ] == frac
                        )
                        & (
                            ACTIVE_LEARNING[
                                "scenario"
                            ] == scenario
                        )
                        & (
                            ACTIVE_LEARNING[
                                "target_fpr"
                            ] == target
                        )
                    ].set_index("seed")

                    common = sorted(
                        set(u.index)
                        & set(r.index)
                    )

                    if common != SEEDS:
                        raise RuntimeError(
                            "AL Seed-Coverage "
                            "unvollständig."
                        )

                    for metric, higher in [
                        ("recall", True),
                        (
                            "empirical_fpr",
                            False,
                        ),
                        ("f1", True),
                        (
                            "average_precision",
                            True,
                        ),
                    ]:
                        diff = (
                            u.loc[
                                common,
                                metric,
                            ].to_numpy()
                            - r.loc[
                                common,
                                metric,
                            ].to_numpy()
                        )

                        al_stat_rows.append({
                            "representation": (
                                rep
                            ),
                            "classifier": (
                                clf
                            ),
                            "label_budget": (
                                frac
                            ),
                            "scenario": (
                                scenario
                            ),
                            "target_fpr": (
                                target
                            ),
                            "metric": (
                                metric
                            ),
                            "contrast": (
                                "UNCERTAINTY_minus_RANDOM"
                            ),
                            **paired_seed_stats(
                                diff,
                                higher_is_better=higher,
                            ),
                        })

AL_STATS = pd.DataFrame(
    al_stat_rows
)

AL_STATS.to_csv(
    FINAL_ROOT
    / "ACTIVE_LEARNING_paired_seed_tests.csv",
    index=False,
)

display(
    AL_SUMMARY[
        AL_SUMMARY[
            "scenario"
        ].eq(
            "DOMAIN_TEMPLATE_OOD"
        )
        & AL_SUMMARY[
            "target_fpr"
        ].eq(
            PRIMARY_TARGET_FPR
        )
    ].head(40)
)

In [ ]:
# ============================================================
# FINAL 15 – Thesis-Tabellen
# ============================================================

# 1: Kernsysteme über alle Labelbudgets
FINAL_SUMMARY[
    FINAL_SUMMARY[
        "model"
    ].isin(FINAL_CORE_SYSTEMS)
    & FINAL_SUMMARY[
        "scenario"
    ].isin(FINAL_STRESS)
    & FINAL_SUMMARY[
        "target_fpr"
    ].eq(PRIMARY_TARGET_FPR)
].to_csv(
    FINAL_TABLE_ROOT
    / "TABLE01_core_systems_all_budgets_fpr005.csv",
    index=False,
)

# 2: Alle Systeme, 25 %, alle FPR-Punkte
FINAL_SUMMARY[
    FINAL_SUMMARY[
        "label_budget"
    ].eq(0.25)
    & FINAL_SUMMARY[
        "scenario"
    ].isin(FINAL_STRESS)
].to_csv(
    FINAL_TABLE_ROOT
    / "TABLE02_all_systems_25pct_all_fpr.csv",
    index=False,
)

# 3: Repräsentation × Downstream-Classifier
FINAL_SUMMARY[
    FINAL_SUMMARY[
        "model"
    ].isin(FINAL_EMBED_SYSTEMS)
    & FINAL_SUMMARY[
        "scenario"
    ].isin(FINAL_STRESS)
    & FINAL_SUMMARY[
        "target_fpr"
    ].eq(PRIMARY_TARGET_FPR)
].to_csv(
    FINAL_TABLE_ROOT
    / "TABLE03_representation_classifier_effect.csv",
    index=False,
)

# 4: AP korrigiert
FINAL_AP_SUMMARY.to_csv(
    FINAL_TABLE_ROOT
    / "TABLE04_corrected_average_precision.csv",
    index=False,
)

# 5: DAPT-E2E Statistik
FINAL_STATS[
    FINAL_STATS[
        "contrast"
    ].isin([
        "DAPT_E2E_minus_T0_E2E",
        "DAPT_E2E_minus_B0",
    ])
    & FINAL_STATS[
        "scenario"
    ].isin(FINAL_STRESS)
].to_csv(
    FINAL_TABLE_ROOT
    / "TABLE05_dapt_e2e_statistics.csv",
    index=False,
)

# 6: Kaskade
FINAL_CASCADE_SUMMARY[
    FINAL_CASCADE_SUMMARY[
        "ranker"
    ].isin([
        "B0_SELF",
        "T0_E2E",
        "DAPT_E2E",
        "DAPT_MLP",
    ])
    & FINAL_CASCADE_SUMMARY[
        "scenario"
    ].isin(FINAL_STRESS)
].to_csv(
    FINAL_TABLE_ROOT
    / "TABLE06_cascade_core.csv",
    index=False,
)

# 7: Fehlerkomplementarität
FINAL_ERROR_SUMMARY[
    FINAL_ERROR_SUMMARY[
        "candidate_model"
    ].isin([
        "T0_E2E",
        "DAPT_E2E",
        "DAPT_MLP",
        "CONTRASTIVE_MLP",
    ])
    & FINAL_ERROR_SUMMARY[
        "scenario"
    ].isin(FINAL_STRESS)
].to_csv(
    FINAL_TABLE_ROOT
    / "TABLE07_error_complementarity.csv",
    index=False,
)

# 8: Active Learning
AL_SUMMARY[
    AL_SUMMARY[
        "scenario"
    ].isin(FINAL_STRESS)
    & AL_SUMMARY[
        "target_fpr"
    ].eq(PRIMARY_TARGET_FPR)
].to_csv(
    FINAL_TABLE_ROOT
    / "TABLE08_active_learning.csv",
    index=False,
)

# 9: Modellranking
FINAL_RANK_SUMMARY.to_csv(
    FINAL_TABLE_ROOT
    / "TABLE09_model_rank_stability.csv",
    index=False,
)

# 10: tatsächliche FPR explizit
FINAL_SUMMARY[
    FINAL_SUMMARY[
        "scenario"
    ].isin(FINAL_STRESS)
][
    [
        "model",
        "label_budget",
        "scenario",
        "target_fpr",
        "empirical_fpr_mean",
        "empirical_fpr_std",
        "recall_mean",
        "recall_std",
    ]
].to_csv(
    FINAL_TABLE_ROOT
    / "TABLE10_fpr_complete.csv",
    index=False,
)

print({
    "thesis_tables": len(
        list(
            FINAL_TABLE_ROOT.glob(
                "*.csv"
            )
        )
    )
})

In [ ]:
# ============================================================
# FINAL 16 – Abbildungs-Helper
# ============================================================

FIGURE_MANIFEST = []

MODEL_LABELS = {
    "B0_STRUCT_XGB": (
        "B0 Struktur-XGB"
    ),
    "T0_E2E": (
        "T0 End-to-End"
    ),
    "DAPT_E2E": (
        "DAPT End-to-End"
    ),
    "BASE_LOGREG": (
        "BASE + LR"
    ),
    "BASE_XGBOOST": (
        "BASE + XGB"
    ),
    "BASE_MLP": (
        "BASE + MLP"
    ),
    "DAPT_LOGREG": (
        "DAPT + LR"
    ),
    "DAPT_XGBOOST": (
        "DAPT + XGB"
    ),
    "DAPT_MLP": (
        "DAPT + MLP"
    ),
    "CONTRASTIVE_LOGREG": (
        "Contrastive + LR"
    ),
    "CONTRASTIVE_XGBOOST": (
        "Contrastive + XGB"
    ),
    "CONTRASTIVE_MLP": (
        "Contrastive + MLP"
    ),
    "B0_SELF": (
        "B0 Self-Ranking"
    ),
}

BUDGET_X = {
    0.10: 10,
    0.15: 15,
    0.20: 20,
    0.25: 25,
    1.00: 100,
}

def save_final_figure(
    fig,
    filename,
    title,
    chapter_hint,
):
    png = FINAL_FIG_ROOT / (
        filename + ".png"
    )
    pdf = FINAL_FIG_ROOT / (
        filename + ".pdf"
    )

    fig.tight_layout()
    fig.savefig(
        png,
        dpi=220,
        bbox_inches="tight",
    )
    fig.savefig(
        pdf,
        bbox_inches="tight",
    )
    plt.close(fig)

    FIGURE_MANIFEST.append({
        "figure": filename,
        "png": png.name,
        "pdf": pdf.name,
        "title": title,
        "chapter_hint": chapter_hint,
    })

In [ ]:
# ============================================================
# FINAL 17 – Abbildungen: Kernsysteme / FPR / AP
# ============================================================

# A: Recall über Labelbudgets, je Stressszenario
for scenario in FINAL_STRESS:
    fig, ax = plt.subplots(
        figsize=(8.2, 5.0)
    )

    data = FINAL_SUMMARY[
        FINAL_SUMMARY[
            "model"
        ].isin(FINAL_CORE_SYSTEMS)
        & FINAL_SUMMARY[
            "scenario"
        ].eq(scenario)
        & FINAL_SUMMARY[
            "target_fpr"
        ].eq(PRIMARY_TARGET_FPR)
    ]

    for model in FINAL_CORE_SYSTEMS:
        p = data[
            data["model"] == model
        ].sort_values(
            "label_budget"
        )

        ax.plot(
            [
                BUDGET_X[x]
                for x in p[
                    "label_budget"
                ]
            ],
            p["recall_mean"],
            marker="o",
            label=MODEL_LABELS[
                model
            ],
        )

    ax.set_xlabel(
        "Gelabelte Trainingsdaten [%]"
    )
    ax.set_ylabel("Recall")
    ax.set_title(
        "Recall bei 0,5 % Ziel-FPR – "
        + scenario
    )
    ax.set_xticks([10, 25, 100])
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.25)

    save_final_figure(
        fig,
        "FIG01_recall_budget_"
        + scenario.lower(),
        "Recall über Labelbudgets – "
        + scenario,
        (
            "Evaluation / "
            "Label-Effizienz und Shift"
        ),
    )

# B: Recall über FPR bei 25 %
for scenario in FINAL_STRESS:
    fig, ax = plt.subplots(
        figsize=(8.2, 5.0)
    )

    data = FINAL_SUMMARY[
        FINAL_SUMMARY[
            "model"
        ].isin(FINAL_CORE_SYSTEMS)
        & FINAL_SUMMARY[
            "scenario"
        ].eq(scenario)
        & FINAL_SUMMARY[
            "label_budget"
        ].eq(0.25)
    ]

    for model in FINAL_CORE_SYSTEMS:
        p = data[
            data["model"] == model
        ].sort_values(
            "target_fpr"
        )

        ax.plot(
            100 * p["target_fpr"],
            p["recall_mean"],
            marker="o",
            label=MODEL_LABELS[
                model
            ],
        )

    ax.set_xlabel(
        "Ziel-FPR auf "
        "Kalibrierungsdaten [%]"
    )
    ax.set_ylabel("Recall")
    ax.set_title(
        "Recall über Betriebspunkte – "
        + scenario
    )
    ax.set_xticks([0.5, 1.0, 2.0])
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.25)

    save_final_figure(
        fig,
        "FIG02_recall_fpr_25pct_"
        + scenario.lower(),
        "Recall über FPR – "
        + scenario,
        (
            "Evaluation / "
            "operative Betriebspunkte"
        ),
    )

# C: empirische FPR, je Labelbudget
for frac in LABEL_BUDGETS:
    fig, ax = plt.subplots(
        figsize=(9.0, 5.0)
    )

    data = FINAL_SUMMARY[
        FINAL_SUMMARY[
            "model"
        ].isin(FINAL_CORE_SYSTEMS)
        & FINAL_SUMMARY[
            "scenario"
        ].isin(FINAL_STRESS)
        & FINAL_SUMMARY[
            "label_budget"
        ].eq(frac)
        & FINAL_SUMMARY[
            "target_fpr"
        ].eq(PRIMARY_TARGET_FPR)
    ]

    x = np.arange(
        len(FINAL_STRESS)
    )

    for model in FINAL_CORE_SYSTEMS:
        p = (
            data[
                data["model"] == model
            ]
            .set_index("scenario")
            .loc[FINAL_STRESS]
        )

        ax.plot(
            x,
            100
            * p[
                "empirical_fpr_mean"
            ],
            marker="o",
            label=MODEL_LABELS[
                model
            ],
        )

    ax.axhline(
        0.5,
        linestyle="--",
        linewidth=1,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(
        FINAL_STRESS,
        rotation=20,
        ha="right",
    )
    ax.set_ylabel(
        "Beobachtete FPR [%]"
    )
    ax.set_title(
        "FPR unter Distribution Shift – "
        f"{int(frac * 100)} % Labels"
    )
    ax.legend()
    ax.grid(True, alpha=0.25)

    save_final_figure(
        fig,
        "FIG03_empirical_fpr_"
        + str(int(frac * 100))
        + "pct",
        (
            "Beobachtete FPR – "
            f"{int(frac * 100)} % Labels"
        ),
        (
            "Evaluation / "
            "Robustheit des Betriebspunkts"
        ),
    )

# D: korrigierter AP bei 25 %
fig, ax = plt.subplots(
    figsize=(9.0, 5.0)
)

ap_scenarios = [
    "IID_BALANCED",
    *FINAL_STRESS,
]

data = FINAL_AP_SUMMARY[
    FINAL_AP_SUMMARY[
        "model"
    ].isin(FINAL_CORE_SYSTEMS)
    & FINAL_AP_SUMMARY[
        "label_budget"
    ].eq(0.25)
]

x = np.arange(
    len(ap_scenarios)
)

for model in FINAL_CORE_SYSTEMS:
    p = (
        data[
            data["model"] == model
        ]
        .set_index("scenario")
        .loc[ap_scenarios]
    )

    ax.plot(
        x,
        p[
            "average_precision_mean"
        ],
        marker="o",
        label=MODEL_LABELS[
            model
        ],
    )

ax.set_xticks(x)
ax.set_xticklabels(
    ap_scenarios,
    rotation=20,
    ha="right",
)
ax.set_ylabel(
    "Average Precision"
)
ax.set_ylim(0, 1)
ax.set_title(
    "Prävalenzkorrigierter AP – "
    "25 % Labels"
)
ax.legend()
ax.grid(True, alpha=0.25)

save_final_figure(
    fig,
    "FIG04_corrected_ap_25pct",
    (
        "Prävalenzkorrigierter "
        "Average-Precision-Vergleich"
    ),
    (
        "Evaluation / "
        "AP unter Distribution Shift"
    ),
)

In [ ]:
# ============================================================
# FINAL 18 – Abbildungen: LR/XGB/MLP + Seeds
# ============================================================

for rep in REPRESENTATIONS:
    fig, ax = plt.subplots(
        figsize=(8.2, 5.0)
    )

    data = CLASSIFIER_ROLE_SUMMARY[
        CLASSIFIER_ROLE_SUMMARY[
            "representation"
        ].eq(rep)
    ]

    for clf in CLASSIFIERS:
        p = data[
            data["classifier"] == clf
        ].sort_values(
            "label_budget"
        )

        ax.plot(
            [
                BUDGET_X[x]
                for x in p[
                    "label_budget"
                ]
            ],
            p[
                "mean_stress_recall"
            ],
            marker="o",
            label=clf,
        )

    ax.set_xlabel(
        "Gelabelte Trainingsdaten [%]"
    )
    ax.set_ylabel(
        "Mittlerer Recall "
        "über Stressszenarien"
    )
    ax.set_xticks([10, 25, 100])
    ax.set_ylim(0, 1)
    ax.set_title(
        "Downstream-Klassifikator – "
        + rep
    )
    ax.legend()
    ax.grid(True, alpha=0.25)

    save_final_figure(
        fig,
        "FIG05_classifier_"
        + rep.lower(),
        (
            "LR, XGBoost und MLP "
            "auf " + rep
        ),
        (
            "Evaluation / "
            "Downstream-Klassifikator"
        ),
    )

# gepaarte Seed-Linien DAPT vs T0 bei 25 %
for scenario in FINAL_STRESS:
    fig, ax = plt.subplots(
        figsize=(7.5, 5.0)
    )

    t0 = FINAL_MASTER[
        (FINAL_MASTER.model == "T0_E2E")
        & (
            FINAL_MASTER.label_budget
            == 0.25
        )
        & (
            FINAL_MASTER.scenario
            == scenario
        )
        & (
            FINAL_MASTER.target_fpr
            == PRIMARY_TARGET_FPR
        )
    ].set_index("seed")

    dapt = FINAL_MASTER[
        (
            FINAL_MASTER.model
            == "DAPT_E2E"
        )
        & (
            FINAL_MASTER.label_budget
            == 0.25
        )
        & (
            FINAL_MASTER.scenario
            == scenario
        )
        & (
            FINAL_MASTER.target_fpr
            == PRIMARY_TARGET_FPR
        )
    ].set_index("seed")

    for seed in SEEDS:
        ax.plot(
            [0, 1],
            [
                t0.loc[
                    seed, "recall"
                ],
                dapt.loc[
                    seed, "recall"
                ],
            ],
            marker="o",
        )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(
        ["T0", "DAPT"]
    )
    ax.set_ylabel("Recall")
    ax.set_ylim(0, 1)
    ax.set_title(
        "Gepaarte Seeds – "
        + scenario
    )
    ax.grid(True, alpha=0.25)

    save_final_figure(
        fig,
        "FIG06_seed_pairs_"
        + scenario.lower(),
        (
            "Seed-Paare DAPT vs T0 – "
            + scenario
        ),
        (
            "Evaluation / "
            "statistische Robustheit"
        ),
    )

In [ ]:
# ============================================================
# FINAL 19 – Abbildungen: Kaskade / Fehler / Active Learning
# ============================================================

for scenario in FINAL_STRESS:
    fig, ax = plt.subplots(
        figsize=(8.5, 5.0)
    )

    data = FINAL_CASCADE_SUMMARY[
        FINAL_CASCADE_SUMMARY[
            "label_budget"
        ].eq(0.25)
        & FINAL_CASCADE_SUMMARY[
            "scenario"
        ].eq(scenario)
        & FINAL_CASCADE_SUMMARY[
            "target_fpr"
        ].eq(PRIMARY_TARGET_FPR)
        & FINAL_CASCADE_SUMMARY[
            "ranker"
        ].isin([
            "B0_SELF",
            "T0_E2E",
            "DAPT_E2E",
            "DAPT_MLP",
        ])
    ]

    for ranker in [
        "B0_SELF",
        "T0_E2E",
        "DAPT_E2E",
        "DAPT_MLP",
    ]:
        p = data[
            data["ranker"] == ranker
        ].sort_values(
            "review_fraction_of_stage1_negatives"
        )

        ax.plot(
            100
            * p[
                "review_fraction_of_stage1_negatives"
            ],
            p[
                "assisted_recall_mean"
            ],
            marker="o",
            label=MODEL_LABELS.get(
                ranker,
                ranker,
            ),
        )

    ax.set_xlabel(
        "Reviewbudget der "
        "B0-negativen Fälle [%]"
    )
    ax.set_ylabel(
        "Potenzieller "
        "Review-gestützter Recall"
    )
    ax.set_xticks([5, 10, 20])
    ax.set_ylim(0, 1)
    ax.set_title(
        "Selektive Kaskade – "
        + scenario
    )
    ax.legend()
    ax.grid(True, alpha=0.25)

    save_final_figure(
        fig,
        "FIG07_cascade_"
        + scenario.lower(),
        "Kaskade – " + scenario,
        "Evaluation / Kaskade",
    )

# Fehlerrescue 25 %
fig, ax = plt.subplots(
    figsize=(9.0, 5.0)
)

candidates = [
    "T0_E2E",
    "DAPT_E2E",
    "DAPT_MLP",
    "CONTRASTIVE_MLP",
]

data = FINAL_ERROR_SUMMARY[
    FINAL_ERROR_SUMMARY[
        "label_budget"
    ].eq(0.25)
    & FINAL_ERROR_SUMMARY[
        "scenario"
    ].isin(FINAL_STRESS)
    & FINAL_ERROR_SUMMARY[
        "target_fpr"
    ].eq(PRIMARY_TARGET_FPR)
    & FINAL_ERROR_SUMMARY[
        "candidate_model"
    ].isin(candidates)
]

x = np.arange(
    len(FINAL_STRESS)
)

for model in candidates:
    p = (
        data[
            data[
                "candidate_model"
            ] == model
        ]
        .set_index("scenario")
        .loc[FINAL_STRESS]
    )

    ax.plot(
        x,
        100
        * p[
            "fn_rescue_rate_mean"
        ],
        marker="o",
        label=MODEL_LABELS.get(
            model,
            model,
        ),
    )

ax.set_xticks(x)
ax.set_xticklabels(
    FINAL_STRESS,
    rotation=20,
    ha="right",
)
ax.set_ylabel(
    "Gerettete B0-FN [%]"
)
ax.set_title(
    "Fehlerkomplementarität – "
    "25 % Labels"
)
ax.legend()
ax.grid(True, alpha=0.25)

save_final_figure(
    fig,
    "FIG08_error_rescue_25pct",
    (
        "Rescue-Rate der "
        "B0-False-Negatives"
    ),
    "Evaluation / Fehleranalyse",
)

# Active Learning – DAPT pro Classifier
for clf in CLASSIFIERS:
    fig, ax = plt.subplots(
        figsize=(8.2, 5.0)
    )

    data = AL_SUMMARY[
        AL_SUMMARY[
            "representation"
        ].eq("DAPT")
        & AL_SUMMARY[
            "classifier"
        ].eq(clf)
        & AL_SUMMARY[
            "scenario"
        ].eq(
            "DOMAIN_TEMPLATE_OOD"
        )
        & AL_SUMMARY[
            "target_fpr"
        ].eq(PRIMARY_TARGET_FPR)
    ]

    for strategy in AL_STRATEGIES:
        p = data[
            data["strategy"]
            == strategy
        ].sort_values(
            "label_budget"
        )

        ax.plot(
            [
                BUDGET_X[x]
                for x in p[
                    "label_budget"
                ]
            ],
            p["recall_mean"],
            marker="o",
            label=strategy,
        )

    ax.set_xlabel(
        "Gelabelte Trainingsdaten [%]"
    )
    ax.set_ylabel("Recall")
    ax.set_xticks(
        [10, 15, 20, 25]
    )
    ax.set_ylim(0, 1)
    ax.set_title(
        "Active Learning – DAPT + "
        + clf
        + " – Domain+Template-OOD"
    )
    ax.legend()
    ax.grid(True, alpha=0.25)

    save_final_figure(
        fig,
        "FIG09_active_learning_dapt_"
        + clf.lower(),
        (
            "Active Learning "
            "DAPT + " + clf
        ),
        (
            "Explorative Erweiterung / "
            "Active Learning"
        ),
    )

In [ ]:
# ============================================================
# FINAL 20 – Abbildungen: Ranking + Figure Manifest
# ============================================================

for frac in LABEL_BUDGETS:
    fig, ax = plt.subplots(
        figsize=(9.5, 6.0)
    )

    p = FINAL_RANK_SUMMARY[
        FINAL_RANK_SUMMARY[
            "label_budget"
        ].eq(frac)
    ].sort_values(
        "mean_rank"
    )

    y_pos = np.arange(len(p))

    ax.barh(
        y_pos,
        p["mean_rank"],
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels([
        MODEL_LABELS.get(
            x, x
        )
        for x in p["model"]
    ])

    ax.invert_yaxis()
    ax.set_xlabel(
        "Mittlerer Recall-Rang "
        "über Seeds und Stressszenarien"
    )
    ax.set_title(
        "Modellranking – "
        f"{int(frac * 100)} % Labels"
    )
    ax.grid(
        True,
        axis="x",
        alpha=0.25,
    )

    save_final_figure(
        fig,
        "FIG10_model_ranking_"
        + str(int(frac * 100))
        + "pct",
        (
            "Mittlerer Modellrang – "
            f"{int(frac * 100)} % Labels"
        ),
        (
            "Evaluation / "
            "Gesamteinordnung"
        ),
    )

pd.DataFrame(
    FIGURE_MANIFEST
).to_csv(
    FINAL_FIG_ROOT
    / "figure_manifest.csv",
    index=False,
)

print({
    "figure_count": len(
        FIGURE_MANIFEST
    ),
    "png": len(
        list(
            FINAL_FIG_ROOT.glob(
                "*.png"
            )
        )
    ),
    "pdf": len(
        list(
            FINAL_FIG_ROOT.glob(
                "*.pdf"
            )
        )
    ),
})

In [ ]:
# ============================================================
# FINAL 21 – Ergebnis-README + Completion + ZIP
# ============================================================

readme_lines = [
    "# FINAL BACHELOR RUN – Ergebnispaket",
    "",
    "## Zentrale Dateien",
    "",
    "- MASTER_threshold_results.csv",
    "- MASTER_threshold_summary.csv",
    "- MASTER_ap_prevalence_matched.csv",
    "- MASTER_ap_equal_n.csv",
    "- AP_SHIFT_prevalence_matched.csv",
    "- AP_SHIFT_equal_n.csv",
    "- STATISTICS_paired_seed_tests.csv",
    "- MODEL_rank_stability_summary.csv",
    "- CASCADE_results.csv",
    "- CASCADE_summary.csv",
    "- ERROR_complementarity.csv",
    "- ERROR_complementarity_summary.csv",
    "- ACTIVE_LEARNING_results.csv",
    "- ACTIVE_LEARNING_summary.csv",
    "- ACTIVE_LEARNING_paired_seed_tests.csv",
    "",
    "## Thesis-Ausgabe",
    "",
    "- thesis_tables/: direkt vorbereitete Ergebnistabellen",
    "- figures/: PNG + PDF + figure_manifest.csv",
    "",
    "## Methodische Fixpunkte",
    "",
    "- Seeds: 42, 52, 62, 72, 82",
    "- Kein Seed wurde ausgeschlossen.",
    "- Ziel-FPRs: 0,5 %, 1 %, 2 %.",
    "- AP-Shift nur gegen IID_BALANCED.",
    "- Equal-N ist zusätzliche Sensitivitätsanalyse.",
    "- Stress: Temporal, Domain-OOD, Template-OOD, Domain+Template-OOD.",
    "- Active Learning ist explorativ und verändert den SSL-Kern nicht.",
]

(FINAL_ROOT / "RESULTS_PACKAGE_README.md").write_text(
    "\n".join(readme_lines) + "\n",
    encoding="utf-8",
)

checks = {
    "master_expected": (
        len(FINAL_ALL_SYSTEMS)
        * len(SEEDS)
        * len(LABEL_BUDGETS)
        * len(SCENARIOS)
        * len(TARGET_FPRS)
    ),
    "master_actual": len(
        FINAL_MASTER
    ),
    "ap_expected": (
        len(FINAL_ALL_SYSTEMS)
        * len(SEEDS)
        * len(LABEL_BUDGETS)
        * 5
    ),
    "ap_actual": len(
        FINAL_AP
    ),
    "active_learning_expected": (
        len(REPRESENTATIONS)
        * len(CLASSIFIERS)
        * len(SEEDS)
        * len(AL_STRATEGIES)
        * len(AL_BUDGETS)
        * len(SCENARIOS)
        * len(TARGET_FPRS)
    ),
    "active_learning_actual": len(
        ACTIVE_LEARNING
    ),
    "thesis_tables": len(
        list(
            FINAL_TABLE_ROOT.glob(
                "*.csv"
            )
        )
    ),
    "figures_png": len(
        list(
            FINAL_FIG_ROOT.glob(
                "*.png"
            )
        )
    ),
    "figures_pdf": len(
        list(
            FINAL_FIG_ROOT.glob(
                "*.pdf"
            )
        )
    ),
}

if (
    checks["master_expected"]
    != checks["master_actual"]
):
    raise RuntimeError(
        "MASTER Completion Check "
        "fehlgeschlagen."
    )

if (
    checks["ap_expected"]
    != checks["ap_actual"]
):
    raise RuntimeError(
        "AP Completion Check "
        "fehlgeschlagen."
    )

if (
    checks[
        "active_learning_expected"
    ]
    != checks[
        "active_learning_actual"
    ]
):
    raise RuntimeError(
        "Active Learning "
        "Completion Check "
        "fehlgeschlagen."
    )

if checks["thesis_tables"] < 10:
    raise RuntimeError(
        "Zu wenige Thesis-Tabellen."
    )

if (
    checks["figures_png"] < 20
    or checks["figures_pdf"] < 20
):
    raise RuntimeError(
        "Zu wenige Thesis-Abbildungen."
    )

completion = {
    "status": "COMPLETE",
    "run": "FINAL_BACHELOR_RUN",
    "systems": FINAL_ALL_SYSTEMS,
    "seeds": SEEDS,
    "seed_exclusions": [],
    "label_budgets": LABEL_BUDGETS,
    "target_fprs": TARGET_FPRS,
    "primary_target_fpr": (
        PRIMARY_TARGET_FPR
    ),
    "scenarios": list(
        SCENARIOS.keys()
    ),
    "ap_reference": (
        "IID_BALANCED"
    ),
    "equal_n_sensitivity": True,
    "statistics": {
        "paired_seed_analysis": True,
        "exact_signflip_test": True,
        "minimum_two_sided_p_n5": (
            2
            / (
                2
                ** len(SEEDS)
            )
        ),
    },
    "active_learning": {
        "status": "exploratory",
        "strategies": (
            AL_STRATEGIES
        ),
        "budgets": AL_BUDGETS,
        "test_data_used_for_selection": (
            False
        ),
    },
    "checks": checks,
}

(FINAL_ROOT / "BACHELOR_FINAL_RUN_COMPLETE.json").write_text(
    json.dumps(
        completion,
        indent=2,
    ),
    encoding="utf-8",
)

archive = shutil.make_archive(
    "/kaggle/working/phreshphish_FINAL_BACHELOR_RUN",
    "zip",
    root_dir=FINAL_ROOT,
)

print(
    json.dumps(
        completion,
        indent=2,
    )
)

print({
    "FINAL_ZIP": archive
})